In [ ]:

# CELL 1 — FINAL LOCKED TEST AUTHORIZATION + IMMUTABLE SCIENTIFIC CONSTANTS
from pathlib import Path
import os, json, hashlib, math, re, shutil, zipfile, tempfile
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

AUTHORIZE_LOCKED_TEST = True
LOCKED_TEST_ACK = "I_UNDERSTAND_THIS_IS_THE_FINAL_LOCKED_TEST"

PUBLICATION_SEEDS = [42, 1337, 2026, 31415, 271828]
MODEL_ID = "openbmb/MiniCPM-V-4_5"
MODEL_REVISION = "daef484c35ec93210ec93c5e901f8f3e9b78ee34"
EXPECTED_HIDDEN_SIZE = 4096
EXPECTED_QUERY_NUM = 64

ARCHITECTURE_NAME = "FacetGate"
PROTOCOL_VERSION = "1.0.1"
CONTROLLER_REVISION = "MiniCPM-FacetGate-v1.0.1-FINAL-LOCKED-TEST-R3-FEATURE-FIRST-FP32-SPECIALIST"
PROTOCOL_SHA256 = "37b7d30a994fb98225d49d76cca5768a838d456abf844dc1a5d93019a0cd15aa"
PARENT_GATED_PROTOCOL_SHA256 = "627bdf5a77ab694581ff590c32f8885c901ebad09e8df86dbab07762caaf95dd"
MINICPM_NATIVE_BINDING_SHA256 = "7b48b0a1abe63fc3c1aa404a96cc362fb1ae3480605bcbbcc1f0098d43d9d32f"
COMMON_FACETGATE_TARGET_SOURCE_SHA256 = "1c14e1b54846f0ffa1a4a21781703daf7c57ec95a6573a3d2fe0bc14c0f67e3b"
TASK_INSTRUCTION_SHA256 = "bfef1e522a9433e1ca0c57a3a1bea49cd890d8428308b7ffdf6b24e4fe51667f"
SCIENTIFIC_REGIME = "REUSE_EXISTING_GATED"

EXPECTED_LABEL_SHA256 = "9cb9afdd88c9a750d3eb6f36aa512dc6ad5b3067aedfac79b138b7fef64a9bb6"
EXPECTED_OCR_SHA256 = "eed35bf77f749e98d9222fd571d149b99c4c6a610e29bc5db5783d3489492c0c"
EXPECTED_SPLIT_SHA256 = "bbcf7e13fc0aab4ac3b5f58a2ebf8880740ed60ad7ecb2cb61910b0497d1b4f0"
EXPECTED_CANONICAL_ROW_SHA256 = "440f8e6982cf4272fc0b6c3e3e6fc9b079a02ce01cd5350a02be0d4a9fa1cf39"
EXPECTED_TEST_IDENTITY_SHA256 = "5f6ac881e9026d682f2f41d65ed1f08e1861d8c8f030bae04d98f9b9db6419db"
EXPECTED_IMAGE_BUNDLE_SHA256 = "9e134f26c63c23ba871c954ee1d7e03499a47ac3f9bd49d12e0c5ffa977665af"

BINARY_TASKS = ["offensive","vulgar","abuse","sarcastic"]
LANGUAGE_LABELS = ["Bangla","English","Banglish"]
TARGET_LABELS = ["Individual","Gender","Political","Social","National","Religious","Others"]
TASK_INSTRUCTION = (
    "Encode this meme for six independent annotations: offensive, vulgar, abuse, sarcastic, "
    "language (Bangla/English/Banglish), and zero or more of seven real target groups. "
    "Do not impose logical implications between tasks. "
)

NATIVE_IMAGE_MARKER = "(<image>./</image>)"
NATIVE_CHAT_ADD_GENERATION_PROMPT = True
NATIVE_CHAT_ENABLE_THINKING = False
MAX_OCR_TOKENS = 384
NATIVE_MAX_SLICE_NUMS = 9
NATIVE_MAX_INPUT_LENGTH = 16384
BNB_4BIT_SKIP_MODULES = ["llm.lm_head","resampler.attn.out_proj"]
RESAMPLER_BATCH_DIM_STATUS = "CORRECT_K_SHAPE_1"

# Engineering provenance only; scientific model/checkpoint/threshold state is unchanged.
# The supplied R2 attempt reached the first TEST backbone forward on each rank and then
# aborted before producing a usable SHTE prediction, ledger, metric, or TEST-derived decision.
PRIOR_R2_TECHNICAL_ABORT_RECORDED = True
PRIOR_R2_WORKER_SHA256 = "4bfaac6424744d6ee7ef423eb3687030b0a52cc237089ba6275f57493eda4fe6"
PRIOR_R2_ABORT_REASON = "SHTE dtype boundary: expected scalar type Half but found Float"
PRIOR_R2_USABLE_TEST_PREDICTIONS = 0
PRIOR_R2_TEST_DERIVED_TUNING = False
PRIOR_R2_PHYSICAL_TEST_FORWARD_UPPER_BOUND = 2

CERTIFIED_PARENT = {
    42:     {"step":932, "checkpoint_sha256":"ffd8ae90ce0130012fd9a99fe28225919af92eedfbd0dc03d0b982789996e2db"},
    1337:   {"step":800, "checkpoint_sha256":"53304788ffc7bb928c303c88cd1ee29d1a8aa868dc85d0d680109bf9e6d2371c"},
    2026:   {"step":800, "checkpoint_sha256":"45bb11afc5064baf325c7a24f83964e2e4b78e4c82dc0fc8e2001e69afd1036f"},
    31415:  {"step":800, "checkpoint_sha256":"651564574b63936786746b67fb120c4036a6829b506a10d74abaa679ad313759"},
    271828: {"step":800, "checkpoint_sha256":"943b832d1d7884b4d2d45a8a0bd2d4070d59f9bab8a7206007404324aff8e11f"},
}
# These are the already-completed, user-supplied MiniCPM FacetGate development endpoints.
# They are used only as fail-closed provenance checks before TEST is opened.
EXPECTED_DEVELOPMENT = {
    42:     {"shte_step":400, "target_delta_pp":0.9850356112650283},
    1337:   {"shte_step":932, "target_delta_pp":9.200605107238585},
    2026:   {"shte_step":800, "target_delta_pp":4.880145027195809},
    31415:  {"shte_step":800, "target_delta_pp":4.5136457224765225},
    271828: {"shte_step":800, "target_delta_pp":7.369124024988988},
}

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/M3Offend_9318_MiniCPM_V45_FacetGate_5Seed_FINAL_LOCKED_TEST_v1_0_1_R3")
TEMP_ROOT = Path("/kaggle/temp/minicpm_v45_facetgate_locked_test")
WORK_ROOT.mkdir(parents=True,exist_ok=True)
TEMP_ROOT.mkdir(parents=True,exist_ok=True)

def sha256_file(path, chunk=8*1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

def load_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def atomic_json(path,obj):
    p=Path(path); p.parent.mkdir(parents=True,exist_ok=True)
    tmp=p.with_suffix(p.suffix+".tmp")
    tmp.write_text(json.dumps(obj,indent=2,ensure_ascii=False,default=str),encoding="utf-8")
    os.replace(tmp,p)

def json_sha(obj):
    return hashlib.sha256(
        json.dumps(obj,ensure_ascii=False,sort_keys=True,separators=(",",":"),default=str).encode("utf-8")
    ).hexdigest()

def flatten_tensors(obj,prefix=""):
    out={}
    if torch.is_tensor(obj):
        out[prefix.rstrip(".")]=obj
    elif isinstance(obj,dict):
        for k,v in obj.items():
            out.update(flatten_tensors(v,f"{prefix}{k}."))
    return out

def canonical_lora_key(k):
    k=str(k)
    for p in ("peft.","backbone."):
        if k.startswith(p): k=k[len(p):]
    k=k.replace(".default.",".")
    while k.startswith("base_model.model.base_model.model."):
        k=k[len("base_model.model."):]
    return k

def lora_module_from_key(k):
    k=canonical_lora_key(k)
    k=re.sub(r"\.lora_[AB]\.weight$","",k)
    if k.startswith("base_model.model."): k=k[len("base_model.model."):]
    return k


class ResidualFacetAdapter(nn.Module):
    def __init__(self, hidden=1024, bottleneck=256, dropout=0.10):
        super().__init__()
        self.norm=nn.LayerNorm(hidden)
        self.block=nn.Sequential(
            nn.Linear(hidden,bottleneck), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(bottleneck,hidden), nn.Dropout(dropout),
        )
    def forward(self,x):
        return x + self.block(self.norm(x))

class FacetGateTarget(nn.Module):
    def __init__(self):
        super().__init__()
        D=1024
        self.product_norm=nn.LayerNorm(D)
        self.difference_norm=nn.LayerNorm(D)
        self.target_view_gate=nn.Sequential(nn.Linear(4*D,128),nn.GELU(),nn.Linear(128,4))
        self.target_expert_shared=nn.Sequential(nn.Linear(D,D),nn.GELU(),nn.Dropout(0.10))
        self.target_expert_adapter=ResidualFacetAdapter(D,256,0.10)
        self.target_expert_head=nn.Linear(D,7)
        self.target_selector=nn.Sequential(nn.Linear(6,16),nn.GELU(),nn.Linear(16,1))

    def forward(self,v,t,J,zG):
        # J is deliberately part of the frozen common interface even though the
        # four-view v1.0 specialist uses v/t/product/difference exactly.
        if v.shape[-1]!=1024 or t.shape[-1]!=1024 or J.shape[-1]!=1024 or zG.shape[-1]!=7:
            raise RuntimeError(f"Facet interface shape drift: v={v.shape} t={t.shape} J={J.shape} zG={zG.shape}")
        prod=self.product_norm(v*t)
        diff=self.difference_norm(torch.abs(v-t))
        views=torch.stack([v,t,prod,diff],dim=1)
        gate_in=torch.cat([v,t,prod,diff],dim=-1)
        view_weights=F.softmax(self.target_view_gate(gate_in.float()),dim=-1)
        h=(views.float()*view_weights.unsqueeze(-1)).sum(dim=1)
        zH=self.target_expert_head(self.target_expert_adapter(self.target_expert_shared(h)))
        pG=torch.sigmoid(zG.float()); pH=torch.sigmoid(zH.float())
        d_it=(1.0-F.cosine_similarity(v.float(),t.float(),dim=-1)).clamp(0,2).unsqueeze(-1).expand(-1,7)
        sf=torch.stack([pG,pH,(pH-pG).abs(),(pG-.5).abs(),(pH-.5).abs(),d_it],dim=-1)
        selector_logit=self.target_selector(sf.reshape(-1,6)).reshape(-1,7)
        selector=torch.sigmoid(selector_logit)
        zStar=zG.float()+selector*(zH.float()-zG.float())
        return zStar,{
            "zH":zH,
            "selector_logit":selector_logit,
            "selector":selector,
            "view_weights":view_weights,
            "image_text_disagreement":d_it[:,0],
        }

_common_source = """class ResidualFacetAdapter(nn.Module):
    def __init__(self, hidden=1024, bottleneck=256, dropout=0.10):
        super().__init__()
        self.norm=nn.LayerNorm(hidden)
        self.block=nn.Sequential(
            nn.Linear(hidden,bottleneck), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(bottleneck,hidden), nn.Dropout(dropout),
        )
    def forward(self,x):
        return x + self.block(self.norm(x))

class FacetGateTarget(nn.Module):
    def __init__(self):
        super().__init__()
        D=1024
        self.product_norm=nn.LayerNorm(D)
        self.difference_norm=nn.LayerNorm(D)
        self.target_view_gate=nn.Sequential(nn.Linear(4*D,128),nn.GELU(),nn.Linear(128,4))
        self.target_expert_shared=nn.Sequential(nn.Linear(D,D),nn.GELU(),nn.Dropout(0.10))
        self.target_expert_adapter=ResidualFacetAdapter(D,256,0.10)
        self.target_expert_head=nn.Linear(D,7)
        self.target_selector=nn.Sequential(nn.Linear(6,16),nn.GELU(),nn.Linear(16,1))

    def forward(self,v,t,J,zG):
        # J is deliberately part of the frozen common interface even though the
        # four-view v1.0 specialist uses v/t/product/difference exactly.
        if v.shape[-1]!=1024 or t.shape[-1]!=1024 or J.shape[-1]!=1024 or zG.shape[-1]!=7:
            raise RuntimeError(f"Facet interface shape drift: v={v.shape} t={t.shape} J={J.shape} zG={zG.shape}")
        prod=self.product_norm(v*t)
        diff=self.difference_norm(torch.abs(v-t))
        views=torch.stack([v,t,prod,diff],dim=1)
        gate_in=torch.cat([v,t,prod,diff],dim=-1)
        view_weights=F.softmax(self.target_view_gate(gate_in.float()),dim=-1)
        h=(views.float()*view_weights.unsqueeze(-1)).sum(dim=1)
        zH=self.target_expert_head(self.target_expert_adapter(self.target_expert_shared(h)))
        pG=torch.sigmoid(zG.float()); pH=torch.sigmoid(zH.float())
        d_it=(1.0-F.cosine_similarity(v.float(),t.float(),dim=-1)).clamp(0,2).unsqueeze(-1).expand(-1,7)
        sf=torch.stack([pG,pH,(pH-pG).abs(),(pG-.5).abs(),(pH-.5).abs(),d_it],dim=-1)
        selector_logit=self.target_selector(sf.reshape(-1,6)).reshape(-1,7)
        selector=torch.sigmoid(selector_logit)
        zStar=zG.float()+selector*(zH.float()-zG.float())
        return zStar,{
            "zH":zH,
            "selector_logit":selector_logit,
            "selector":selector,
            "view_weights":view_weights,
            "image_text_disagreement":d_it[:,0],
        }"""
if hashlib.sha256(_common_source.encode("utf-8")).hexdigest()!=COMMON_FACETGATE_TARGET_SOURCE_SHA256:
    raise RuntimeError("FacetGate Target-specialist source hash drift.")

if hashlib.sha256(TASK_INSTRUCTION.encode("utf-8")).hexdigest()!=TASK_INSTRUCTION_SHA256:
    raise RuntimeError("Task-instruction hash drift.")

if not AUTHORIZE_LOCKED_TEST or LOCKED_TEST_ACK!="I_UNDERSTAND_THIS_IS_THE_FINAL_LOCKED_TEST":
    raise RuntimeError("Final locked TEST authorization missing.")

print({
    "architecture":ARCHITECTURE_NAME,
    "model":MODEL_ID,
    "revision":MODEL_REVISION,
    "protocol":"FacetGate v1.0.1",
    "protocol_sha256":PROTOCOL_SHA256,
    "publication_seeds":PUBLICATION_SEEDS,
    "locked_TEST_authorized":True,
})


In [ ]:

# CELL 2 — PIN THE SUCCESSFUL MiniCPM-V-4.5 RUNTIME FAMILY
# Execution-only environment reproduction. Scientific settings are unchanged.
import importlib.metadata as md, subprocess, sys
from pathlib import Path

protected = ["torch","torchvision","numpy","pandas","scikit-learn","scipy","Pillow"]
before={}
for pkg in protected:
    try: before[pkg]=md.version(pkg)
    except Exception: pass

constraints=WORK_ROOT/"minicpm_v45_facetgate_locked_test_constraints.txt"
constraints.write_text("\n".join(f"{k}=={v}" for k,v in before.items())+"\n",encoding="utf-8")

cmd=[
    sys.executable,"-m","pip","install","--no-cache-dir",
    "--upgrade-strategy","only-if-needed","-c",str(constraints),
    "transformers==4.51.0","peft==0.15.2","accelerate==1.6.0","bitsandbytes==0.46.1"
]
print("[PIP]"," ".join(cmd),flush=True)
subprocess.run(cmd,check=True)

after={}
for pkg in protected:
    try: after[pkg]=md.version(pkg)
    except Exception: pass
if before!=after:
    raise RuntimeError(f"Protected base-package drift: before={before} after={after}")

print("[RUNTIME PASS]",after)


In [ ]:

# CELL 3 — DISCOVER, VERIFY, AND FREEZE ALL FIVE COMPLETED MiniCPM FACETGATE DEVELOPMENT PACKAGES
# CPU only. TEST remains unopened.

import os, json, zipfile, hashlib, shutil, re
from pathlib import Path
import numpy as np, torch

EXTRACT_ROOT=TEMP_ROOT/"development_packages"
EXTRACT_ROOT.mkdir(parents=True,exist_ok=True)

def _safe_member(name):
    p=Path(name)
    return not p.is_absolute() and ".." not in p.parts

def _extract_relevant_archives():
    roots=[]
    seen=set()
    archives=[]
    for p in INPUT_ROOT.rglob("*.zip"):
        try:
            if p.is_file(): archives.append(p)
        except OSError:
            pass
    # Two passes support an outer ZIP containing the actual development-results ZIP.
    current=list(archives)
    for depth in range(2):
        nxt=[]
        for zpath in current:
            try:
                zsha=sha256_file(zpath)
                if zsha in seen: continue
                seen.add(zsha)
                with zipfile.ZipFile(zpath,"r") as z:
                    names=z.namelist()
                    interesting=any(
                        n.endswith("FACETGATE_VALIDATION_SCORECARD.json") or
                        n.endswith("FACETGATE_FINAL_COMPOSITE_BINDING.json")
                        for n in names
                    )
                    inner=[n for n in names if n.lower().endswith(".zip") and ("facetgate" in n.lower() or "result" in n.lower())]
                    if interesting:
                        dest=EXTRACT_ROOT/f"zip_{zsha[:16]}"
                        dest.mkdir(parents=True,exist_ok=True)
                        for n in names:
                            if not _safe_member(n): continue
                            low=n.lower()
                            keep=(
                                low.endswith("facetgate_validation_scorecard.json") or
                                low.endswith("facetgate_thresholds.json") or
                                low.endswith("facetgate_final_composite_binding.json") or
                                low.endswith("facetgate_controller_complete.json") or
                                low.endswith("facetgate_development_complete.json") or
                                low.endswith("test_forward_counter.json") or
                                ("checkpoints/" in low and low.endswith(".pt"))
                            )
                            if keep:
                                z.extract(n,dest)
                        roots.append(dest)
                    for n in inner:
                        if not _safe_member(n): continue
                        info=z.getinfo(n)
                        if info.file_size>3_000_000_000: continue
                        b=z.read(n)
                        ish=hashlib.sha256(b).hexdigest()
                        ip=EXTRACT_ROOT/f"inner_{ish[:16]}.zip"
                        if not ip.exists(): ip.write_bytes(b)
                        nxt.append(ip)
            except Exception:
                continue
        current=nxt
    return roots

EXTRACTED_ROOTS=_extract_relevant_archives()

def _all_scorecards():
    found=[]
    for root in [INPUT_ROOT,*EXTRACTED_ROOTS]:
        try:
            found.extend(root.rglob("FACETGATE_VALIDATION_SCORECARD.json"))
        except Exception:
            pass
    # Deduplicate by resolved path.
    out=[]; seen=set()
    for p in found:
        try: key=str(p.resolve())
        except Exception: key=str(p)
        if key not in seen:
            seen.add(key); out.append(p)
    return out

def _package_root(score_path):
    p=Path(score_path)
    return p.parent.parent if p.parent.name=="artifacts" else p.parent

def _single_under(root,name):
    hits=list(Path(root).rglob(name))
    if not hits: return None
    # If duplicate byte-identical copies exist under the same extracted package, use the shortest path.
    by={}
    for p in hits:
        try: by.setdefault(sha256_file(p),[]).append(p)
        except Exception: pass
    if len(by)!=1:
        return None
    return sorted(next(iter(by.values())),key=lambda x:len(str(x)))[0]

EXPECTED_SHTE_STATE=FacetGateTarget().state_dict()

def _inspect_bundle(score_path):
    score_path=Path(score_path)
    root=_package_root(score_path)
    try:
        score=load_json(score_path)
    except Exception as e:
        return None, f"scorecard_read:{e!r}"
    try:
        if score.get("architecture")!="FacetGate":
            return None,"architecture"
        seed=int(score["seed"])
        if seed not in PUBLICATION_SEEDS:
            return None,"seed"
        if score.get("execution_mode")!=SCIENTIFIC_REGIME:
            return None,f"execution_mode:{score.get('execution_mode')}"
        if int(score.get("locked_TEST_forwards",score.get("test_forward_count",0)))!=0:
            return None,"TEST_already_touched"
        exp=EXPECTED_DEVELOPMENT[seed]
        if int(score["selected_internal_gated_step"])!=CERTIFIED_PARENT[seed]["step"]:
            return None,"gated_step"
        if int(score["selected_internal_shte_step"])!=int(exp["shte_step"]):
            return None,"shte_step"
        if abs(float(score["target_delta_pp"])-float(exp["target_delta_pp"]))>1e-9:
            return None,"development_delta"
        ident=score.get("core5_identity",{})
        if not bool(ident.get("pass",False)) or float(ident.get("max_abs_change",1.0))>1e-10:
            return None,"core5_identity"
        if int(score.get("paired_bootstrap_resamples",0))!=10000:
            return None,"bootstrap_count"
        if float(score.get("target_mAP_delta",-999)) < -1e-12:
            return None,"target_mAP_regression"
        if ((float(score.get("rare_target_macro_f1",0))-float(score.get("rare_target_baseline_macro_f1",0)))*100.0) < -0.5-1e-12:
            return None,"rare_target_guard"

        threshold_path=_single_under(root,"FACETGATE_THRESHOLDS.json")
        binding_path=_single_under(root,"FACETGATE_FINAL_COMPOSITE_BINDING.json")
        controller_path=_single_under(root,"FACETGATE_CONTROLLER_COMPLETE.json")
        marker_path=_single_under(root,"FACETGATE_DEVELOPMENT_COMPLETE.json")
        if threshold_path is None or binding_path is None:
            return None,"missing_threshold_or_binding"

        thresholds=load_json(threshold_path)
        binding=load_json(binding_path)
        if binding.get("architecture")!="FacetGate" or binding.get("model_id")!=MODEL_ID or binding.get("model_revision")!=MODEL_REVISION:
            return None,"binding_model"
        if int(binding.get("seed",-1))!=seed or binding.get("protocol_sha256")!=PROTOCOL_SHA256:
            return None,"binding_protocol_seed"
        if binding.get("execution_mode")!=SCIENTIFIC_REGIME:
            return None,"binding_regime"
        if int(binding.get("selected_internal_gated_step",-1))!=CERTIFIED_PARENT[seed]["step"]:
            return None,"binding_gated_step"
        if int(binding.get("selected_internal_shte_step",-1))!=exp["shte_step"]:
            return None,"binding_shte_step"
        if binding.get("minicpm_gated_adapter_serialization")!="norm_fc1_fc2":
            return None,"binding_adapter_serialization"
        if binding.get("target_specialist_class")!="FacetGateTarget":
            return None,"binding_target_class"
        if int(binding.get("locked_TEST_forwards",0))!=0:
            return None,"binding_TEST"

        # Threshold artifact must be exactly the thresholds archived in the scorecard.
        def fd(x): return {str(k):float(v) for k,v in dict(x).items()}
        if fd(thresholds.get("core5",{}))!=fd(score.get("core5_thresholds",{})):
            return None,"core_threshold_drift"
        if fd(thresholds.get("target",{}))!=fd(score.get("target_thresholds",{})):
            return None,"target_threshold_drift"
        if fd(thresholds.get("internal_gated_target_baseline",{}))!=fd(score.get("internal_gated_target_thresholds",{})):
            return None,"baseline_threshold_drift"
        if set(thresholds["core5"])!=set(BINARY_TASKS):
            return None,"core_threshold_keys"
        if set(thresholds["target"])!=set(TARGET_LABELS):
            return None,"target_threshold_keys"
        if set(thresholds["internal_gated_target_baseline"])!=set(TARGET_LABELS):
            return None,"baseline_threshold_keys"

        step=int(exp["shte_step"])
        ck_hits=list(root.rglob(f"facetgate_internal_shte_candidate_step_{step}.pt"))
        if not ck_hits:
            return None,"missing_selected_shte_checkpoint"
        ck_by={}
        for p in ck_hits:
            try: ck_by.setdefault(sha256_file(p),[]).append(p)
            except Exception: pass
        if len(ck_by)!=1:
            return None,f"ambiguous_selected_shte_checkpoint:{list(ck_by)}"
        shte_sha=next(iter(ck_by))
        shte_path=sorted(ck_by[shte_sha],key=lambda x:len(str(x)))[0]
        if binding.get("selected_internal_shte_sha256")!=shte_sha:
            return None,"binding_shte_sha"
        st=torch.load(shte_path,map_location="cpu",weights_only=False)
        if not isinstance(st,dict) or int(st.get("step",-1))!=step:
            return None,"shte_state_step"
        if st.get("protocol_sha256")!=PROTOCOL_SHA256 or not isinstance(st.get("state_dict"),dict):
            return None,"shte_state_protocol"
        ssd=st["state_dict"]
        if set(ssd)!=set(EXPECTED_SHTE_STATE):
            return None,"shte_state_keys"
        for k,v in EXPECTED_SHTE_STATE.items():
            if tuple(ssd[k].shape)!=tuple(v.shape):
                return None,f"shte_shape:{k}"

        # Modern durable marker: require exact cryptographic closure.
        marker=None
        if marker_path is not None:
            marker=load_json(marker_path)
            if marker.get("status")!="DEVELOPMENT_COMPLETE_TEST_LOCKED":
                return None,"marker_status"
            if marker.get("architecture")!="FacetGate" or marker.get("model_id")!=MODEL_ID or marker.get("model_revision")!=MODEL_REVISION:
                return None,"marker_model"
            if int(marker.get("seed",-1))!=seed or marker.get("protocol_sha256")!=PROTOCOL_SHA256:
                return None,"marker_protocol"
            if int(marker.get("locked_TEST_forwards",-1))!=0:
                return None,"marker_TEST"
            checks={
                "scorecard_sha256":sha256_file(score_path),
                "selected_internal_shte_sha256":shte_sha,
                "final_composite_binding_sha256":sha256_file(binding_path),
                "thresholds_sha256":sha256_file(threshold_path),
            }
            for k,v in checks.items():
                if marker.get(k)!=v:
                    return None,f"marker_hash:{k}"
        else:
            # Legacy completed seeds predate the durable marker. Require their
            # controller-complete artifact plus all scientific artifacts above.
            if controller_path is None:
                return None,"legacy_missing_controller_complete"
            ctl=load_json(controller_path)
            if ctl.get("architecture")!="FacetGate" or int(ctl.get("seed",-1))!=seed:
                return None,"legacy_controller_identity"
            if ctl.get("protocol_sha256")!=PROTOCOL_SHA256 or int(ctl.get("locked_TEST_forwards",0))!=0:
                return None,"legacy_controller_protocol_TEST"

        return {
            "seed":seed,
            "root":str(root),
            "scorecard_path":str(score_path),
            "scorecard_sha256":sha256_file(score_path),
            "threshold_path":str(threshold_path),
            "threshold_sha256":sha256_file(threshold_path),
            "binding_path":str(binding_path),
            "binding_sha256":sha256_file(binding_path),
            "marker_path":None if marker_path is None else str(marker_path),
            "package_layout":"DURABLE_MARKER" if marker_path is not None else "LEGACY_COMPLETE",
            "shte_step":step,
            "shte_checkpoint_path":str(shte_path),
            "shte_checkpoint_sha256":shte_sha,
            "gated_step":CERTIFIED_PARENT[seed]["step"],
            "gated_checkpoint_sha256":CERTIFIED_PARENT[seed]["checkpoint_sha256"],
            "scorecard":score,
            "thresholds":thresholds,
            "binding":binding,
        }, None
    except Exception as e:
        return None,repr(e)

BUNDLE_CANDIDATES={s:[] for s in PUBLICATION_SEEDS}
BUNDLE_REJECTIONS=[]
for p in _all_scorecards():
    b,reason=_inspect_bundle(p)
    if b is None:
        BUNDLE_REJECTIONS.append({"scorecard":str(p),"reason":reason})
    else:
        BUNDLE_CANDIDATES[b["seed"]].append(b)

BUNDLES={}
for seed in PUBLICATION_SEEDS:
    cands=BUNDLE_CANDIDATES[seed]
    # Deduplicate exact copies of the same frozen package.
    uniq={}
    for b in cands:
        key=(b["scorecard_sha256"],b["shte_checkpoint_sha256"],b["threshold_sha256"],b["binding_sha256"])
        uniq.setdefault(key,b)
    if len(uniq)!=1:
        raise RuntimeError(
            f"Seed {seed}: expected exactly one unique completed FacetGate package; "
            f"found={len(uniq)} raw_candidates={len(cands)}. "
            f"Rejections={BUNDLE_REJECTIONS[:20]}"
        )
    BUNDLES[seed]=next(iter(uniq.values()))

# ------------------------------------------------------------------
# Exact same-seed certified Gated parent checkpoint discovery.
# ------------------------------------------------------------------
def _candidate_parent_files(step):
    hits=[]
    for root in [INPUT_ROOT,*EXTRACTED_ROOTS]:
        try:
            hits.extend(root.rglob(f"candidate_step_{step}.pt"))
        except Exception:
            pass
    return hits

PARENTS={}
for seed in PUBLICATION_SEEDS:
    expected=CERTIFIED_PARENT[seed]
    exact=[]
    for p in _candidate_parent_files(expected["step"]):
        try:
            if sha256_file(p)==expected["checkpoint_sha256"]:
                exact.append(p)
        except Exception:
            pass
    # If direct files are absent, look inside any mounted ZIP by member name.
    if not exact:
        for zpath in INPUT_ROOT.rglob("*.zip"):
            try:
                with zipfile.ZipFile(zpath,"r") as z:
                    for n in z.namelist():
                        if n.endswith(f"candidate_step_{expected['step']}.pt") and _safe_member(n):
                            info=z.getinfo(n)
                            if info.file_size>3_000_000_000: continue
                            data=z.read(n)
                            if hashlib.sha256(data).hexdigest()==expected["checkpoint_sha256"]:
                                dest=EXTRACT_ROOT/f"parent_seed{seed}_{expected['checkpoint_sha256'][:16]}.pt"
                                if not dest.exists(): dest.write_bytes(data)
                                exact.append(dest)
            except Exception:
                continue
    if not exact:
        raise FileNotFoundError(
            f"Seed {seed}: exact certified Gated parent checkpoint not found. "
            f"Required step={expected['step']} SHA={expected['checkpoint_sha256']}. "
            "Mount the completed same-seed MiniCPM Gated parent output."
        )
    p=exact[0]
    st=torch.load(p,map_location="cpu",weights_only=False)
    if not isinstance(st,dict) or int(st.get("step",-1))!=expected["step"]:
        raise RuntimeError(f"Seed {seed}: Gated parent step/schema drift.")
    if not isinstance(st.get("peft"),dict) or not isinstance(st.get("classifier"),dict):
        raise RuntimeError(f"Seed {seed}: certified parent must contain peft + classifier.")
    pe=flatten_tensors(st["peft"]); cl=flatten_tensors(st["classifier"])
    lora_modules={lora_module_from_key(k) for k in pe if ".lora_A" in canonical_lora_key(k)}
    if len(pe)!=506 or len(lora_modules)!=253 or len(cl)!=64:
        raise RuntimeError(
            f"Seed {seed}: certified parent serialization drift "
            f"peft={len(pe)} lora_modules={len(lora_modules)} classifier={len(cl)}"
        )
    PARENTS[seed]={"path":str(p),"step":expected["step"],"sha256":expected["checkpoint_sha256"]}

    # Development binding must self-bind the same exact Gated parent.
    bound=BUNDLES[seed]["binding"].get("selected_internal_gated_sha256")
    if bound is not None and bound!=expected["checkpoint_sha256"]:
        raise RuntimeError(f"Seed {seed}: development binding points to a different Gated parent SHA.")

# ------------------------------------------------------------------
# Five-seed development authorization gate. TEST still unopened.
# ------------------------------------------------------------------
deltas=np.asarray([float(BUNDLES[s]["scorecard"]["target_delta_pp"]) for s in PUBLICATION_SEEDS],dtype=float)
map_deltas=np.asarray([float(BUNDLES[s]["scorecard"]["target_mAP_delta"]) for s in PUBLICATION_SEEDS],dtype=float)
core_identity=[bool(BUNDLES[s]["scorecard"]["core5_identity"]["pass"]) for s in PUBLICATION_SEEDS]
rare_guard=[
    (float(BUNDLES[s]["scorecard"]["rare_target_macro_f1"])-float(BUNDLES[s]["scorecard"]["rare_target_baseline_macro_f1"]))*100.0 >= -0.5-1e-12
    for s in PUBLICATION_SEEDS
]
FIVE_SEED_GATE={
    "mean_target_macro_f1_delta_ge_1pp":float(deltas.mean())>=1.0,
    "positive_target_delta_at_least_4_of_5":int((deltas>0).sum())>=4,
    "core5_identity_all_5":all(core_identity),
    "target_mAP_no_decrease_all_5":bool(np.all(map_deltas>=-1e-12)),
    "rare_target_guard_all_5":all(rare_guard),
    "same_protocol_all_5":all(BUNDLES[s]["binding"]["protocol_sha256"]==PROTOCOL_SHA256 for s in PUBLICATION_SEEDS),
    "same_scientific_regime_all_5":all(BUNDLES[s]["binding"]["execution_mode"]==SCIENTIFIC_REGIME for s in PUBLICATION_SEEDS),
    "TEST_forwards_zero_all_5":all(int(BUNDLES[s]["scorecard"].get("locked_TEST_forwards",0))==0 for s in PUBLICATION_SEEDS),
}
if not all(FIVE_SEED_GATE.values()):
    raise RuntimeError(f"Five-seed FacetGate development gate did not pass: {FIVE_SEED_GATE}")

DEV_GATE_REPORT={
    "architecture":"FacetGate",
    "protocol_version":"1.0.1",
    "protocol_sha256":PROTOCOL_SHA256,
    "model_id":MODEL_ID,
    "model_revision":MODEL_REVISION,
    "seeds":PUBLICATION_SEEDS,
    "per_seed_target_delta_pp":{str(s):float(BUNDLES[s]["scorecard"]["target_delta_pp"]) for s in PUBLICATION_SEEDS},
    "mean_target_delta_pp":float(deltas.mean()),
    "positive_seeds":int((deltas>0).sum()),
    "selected_shte_steps":{str(s):int(BUNDLES[s]["shte_step"]) for s in PUBLICATION_SEEDS},
    "gate":FIVE_SEED_GATE,
    "locked_TEST_forwards_before_final_test":0,
}
atomic_json(WORK_ROOT/"FIVE_SEED_DEVELOPMENT_GATE.json",DEV_GATE_REPORT)
FROZEN_BINDINGS={
    str(s):{
        "package_layout":BUNDLES[s]["package_layout"],
        "scorecard_sha256":BUNDLES[s]["scorecard_sha256"],
        "threshold_sha256":BUNDLES[s]["threshold_sha256"],
        "binding_sha256":BUNDLES[s]["binding_sha256"],
        "selected_internal_gated_step":BUNDLES[s]["gated_step"],
        "gated_checkpoint_sha256":PARENTS[s]["sha256"],
        "selected_internal_shte_step":BUNDLES[s]["shte_step"],
        "shte_checkpoint_sha256":BUNDLES[s]["shte_checkpoint_sha256"],
        "core5_thresholds":BUNDLES[s]["thresholds"]["core5"],
        "target_thresholds":BUNDLES[s]["thresholds"]["target"],
        "diagnostic_gated_target_thresholds":BUNDLES[s]["thresholds"]["internal_gated_target_baseline"],
        "locked_TEST_forwards_before_final_test":0,
    } for s in PUBLICATION_SEEDS
}
atomic_json(WORK_ROOT/"FROZEN_FIVE_SEED_BINDINGS.json",FROZEN_BINDINGS)
print("[FIVE-SEED DEVELOPMENT GATE PASS — TEST STILL UNOPENED]")
print(json.dumps(DEV_GATE_REPORT,indent=2))


In [ ]:
# CELL 4 — CANONICAL 9,318 DATASET + ORDERED LOCKED TEST AUDIT
# CPU only. No model forward.
import ast, re, json, hashlib
from pathlib import Path
import numpy as np
import pandas as pd

def read_table(path):
    p=Path(path)
    if p.suffix.lower()==".csv":
        return pd.read_csv(p)
    if p.suffix.lower() in {".jsonl",".ndjson"}:
        return pd.read_json(p, lines=True)
    obj=json.loads(p.read_text(encoding="utf-8"))
    if isinstance(obj,list):
        return pd.DataFrame(obj)
    if isinstance(obj,dict):
        for k in ["data","records","items","annotations","samples"]:
            if isinstance(obj.get(k),list):
                return pd.DataFrame(obj[k])
        if obj and all(isinstance(v,dict) for v in obj.values()):
            return pd.DataFrame([{"__dict_key__":k,**v} for k,v in obj.items()])
    raise ValueError(f"Unsupported table structure: {p}")

def resolve_col(df,cands,what,required=True):
    low={str(c).casefold():c for c in df.columns}
    for c in cands:
        if c.casefold() in low:
            return low[c.casefold()]
    if required:
        raise KeyError(f"Cannot resolve {what}; columns={list(df.columns)}")
    return None

def image_name(v):
    return str(v).replace("\\","/").split("/")[-1]

def norm_key(v):
    s=image_name(v).strip().casefold()
    return re.sub(r"\.(jpg|jpeg|png|webp|bmp)$","",s)

def norm_binary(v,task):
    if isinstance(v,(bool,np.bool_)):
        return "Yes" if bool(v) else "No"
    s=str(v).strip().casefold()
    if s in {"yes","y","1","true","positive","pos"}: return "Yes"
    if s in {"no","n","0","false","negative","neg"}: return "No"
    raise ValueError(f"{task}: bad binary label {v!r}")

def norm_language(v):
    s=" ".join(str(v).strip().split()).casefold()
    aliases={
        "bangla":"Bangla","bengali":"Bangla","english":"English",
        "banglish":"Banglish","banglish (bangla + english)":"Banglish",
        "banglish(bangla + english)":"Banglish","bangla + english":"Banglish",
        "bangla+english":"Banglish","bengali + english":"Banglish",
        "romanized bangla":"Banglish","romanised bangla":"Banglish",
    }
    if s not in aliases: raise ValueError(f"bad language {v!r}")
    return aliases[s]

def norm_target(v):
    if v is None or (isinstance(v,float) and np.isnan(v)):
        vals=[]
    elif isinstance(v,(list,tuple,set)):
        vals=list(v)
    elif isinstance(v,str):
        s=v.strip()
        if not s:
            vals=[]
        else:
            try:
                q=json.loads(s); vals=q if isinstance(q,list) else [q]
            except Exception:
                try:
                    q=ast.literal_eval(s); vals=list(q) if isinstance(q,(list,tuple,set)) else [q]
                except Exception:
                    vals=re.split(r"[,;|]",s)
    else:
        vals=[v]
    canon={x.casefold():x for x in TARGET_LABELS}; canon["other"]="Others"
    out=[]
    for x in vals:
        sx=str(x).strip()
        if not sx or sx.casefold() in {"none","no target","null","nan"}: continue
        if sx.casefold() not in canon: raise ValueError(f"bad target {x!r}")
        c=canon[sx.casefold()]
        if c not in out: out.append(c)
    return [x for x in TARGET_LABELS if x in out]

def parse_labelstudio(df):
    rows=[]; errors=[]
    for rn,r in df.iterrows():
        try:
            data=r.get("data",{})
            if isinstance(data,str): data=json.loads(data)
            image_value=None
            if isinstance(data,dict):
                for k in ["image","image_id","image_name","filename","file_name","source_image_name"]:
                    if data.get(k) not in [None,""]:
                        image_value=data[k]; break
            if image_value is None:
                for k in ["image","image_id","image_name","filename","file_name","source_image_name","id"]:
                    if k in r and pd.notna(r[k]):
                        image_value=r[k]; break
            anns=r.get("annotations",[])
            if isinstance(anns,str): anns=json.loads(anns)
            if not isinstance(anns,list) or not anns: raise ValueError("no annotations")
            coll={k:[] for k in BINARY_TASKS+["language","target"]}
            for ann in anns:
                for res in (ann.get("result",[]) if isinstance(ann,dict) else []):
                    if not isinstance(res,dict): continue
                    frm=str(res.get("from_name","")).casefold()
                    val=res.get("value",{}) or {}
                    choices=val.get("choices",val.get("labels",[]))
                    if isinstance(choices,str): choices=[choices]
                    task=next((k for k in coll if k in frm),None)
                    if task and choices: coll[task].append(list(choices))
            item={"record_key":norm_key(image_value),"label_source_id":str(image_value)}
            for t in BINARY_TASKS:
                vals=[norm_binary(b[0],t) for b in coll[t] if len(b)==1]
                if not vals or len(set(vals))!=1: raise ValueError(f"{t} non-unanimous/missing {vals}")
                item[t]=vals[0]
            lv=[norm_language(b[0]) for b in coll["language"] if len(b)==1]
            if not lv or len(set(lv))!=1: raise ValueError(f"language non-unanimous/missing {lv}")
            item["language"]=lv[0]
            tv=[tuple(norm_target(b)) for b in coll["target"] if b]
            if not tv: tv=[tuple()]
            if len(set(tv))!=1: raise ValueError(f"target non-unanimous {tv}")
            item["target_list"]=list(tv[0]); rows.append(item)
        except Exception as e:
            errors.append({"row":int(rn),"error":repr(e)})
    if errors: raise RuntimeError(f"Label parse errors={len(errors)} first={errors[:3]}")
    return pd.DataFrame(rows)

def parse_labels(path):
    df=read_table(path).copy()
    if "annotations" in df.columns:
        out=parse_labelstudio(df)
    else:
        idc=resolve_col(df,["image_id","image_name","source_image_name","source_image","image","filename","file_name","path","name","id","__dict_key__"],"label id")
        out=pd.DataFrame({"record_key":df[idc].map(norm_key),"label_source_id":df[idc].astype(str)})
        for t in BINARY_TASKS:
            c=resolve_col(df,[t],t); out[t]=df[c].map(lambda x:norm_binary(x,t))
        lc=resolve_col(df,["language","lang"],"language"); out["language"]=df[lc].map(norm_language)
        tc=resolve_col(df,["target","targets"],"target"); out["target_list"]=df[tc].map(norm_target)
    if out.record_key.duplicated().any(): raise RuntimeError("duplicate label IDs")
    return out

def parse_ocr(path):
    df=read_table(path).copy()
    idc=resolve_col(df,["image_id","image_name","source_image_name","source_image","image","filename","file_name","path","name","id","__dict_key__"],"OCR id")
    tc=resolve_col(df,["cleaned_ocr","ocr_cleaned","ocr","text","ocr_text","clean_text"],"OCR text")
    out=pd.DataFrame({"record_key":df[idc].map(norm_key),"cleaned_ocr":df[tc].fillna("").astype(str)})
    if out.record_key.duplicated().any(): raise RuntimeError("duplicate OCR IDs")
    return out

def parse_manifest(path):
    df=read_table(path).copy()
    idc=resolve_col(df,["image_id","image","image_name","filename","file_name","source_image_name"],"manifest id")
    sc=resolve_col(df,["split","set","partition","dataset_split"],"split")
    ic=resolve_col(df,["official_row_index","row_index","index"],"row index",False)
    out=pd.DataFrame({
        "record_key":df[idc].map(norm_key),
        "image_id":df[idc].map(image_name),
        "split":df[sc].astype(str).str.strip().str.casefold().replace({"val":"validation","valid":"validation","dev":"validation"}),
        "official_row_index":np.arange(len(df),dtype=int) if ic is None else pd.to_numeric(df[ic],errors="raise").astype(int),
    })
    if out.record_key.duplicated().any() or out.official_row_index.duplicated().any():
        raise RuntimeError("manifest IDs/indices not unique")
    return out.sort_values("official_row_index",kind="stable").reset_index(drop=True)

metadata=[]
for p in INPUT_ROOT.rglob("*"):
    try:
        if p.is_file() and p.suffix.lower() in {".json",".jsonl",".ndjson",".csv"} and p.stat().st_size<250_000_000:
            metadata.append(p)
    except OSError:
        pass

wanted={"labels":EXPECTED_LABEL_SHA256,"ocr":EXPECTED_OCR_SHA256,"split":EXPECTED_SPLIT_SHA256}
found={}
for i,p in enumerate(metadata,1):
    h=sha256_file(p)
    for role,expected in wanted.items():
        if h==expected: found[role]=p
    if i%100==0: print(f"[META SHA] {i}/{len(metadata)}",flush=True)

if set(found)!=set(wanted):
    raise FileNotFoundError(f"Canonical metadata not found by SHA. Found={found}")

labels=parse_labels(found["labels"])
ocr=parse_ocr(found["ocr"])
manifest=parse_manifest(found["split"])
master=manifest.merge(labels,on="record_key",validate="one_to_one").merge(ocr,on="record_key",validate="one_to_one")

if len(master)!=9318: raise RuntimeError(f"aligned rows={len(master)} != 9318")
counts={k:int(v) for k,v in master.split.value_counts().to_dict().items()}
if counts!={"train":7454,"validation":932,"test":932}: raise RuntimeError(f"split-count drift {counts}")

payload="\n".join(
    f"{int(r.official_row_index)}\t{r.image_id}\t{r.split}"
    for r in master.sort_values("official_row_index").itertuples()
)+"\n"
row_sha=hashlib.sha256(payload.encode()).hexdigest()
if row_sha!=EXPECTED_CANONICAL_ROW_SHA256: raise RuntimeError(f"canonical-row SHA drift {row_sha}")

test_payload=[]
for r in master[master.split.eq("test")].sort_values("official_row_index").to_dict("records"):
    tar=[x for x in TARGET_LABELS if x in set(r["target_list"])]
    test_payload.append({
        "sample_id":str(int(r["official_row_index"])),
        "image":Path(str(r["image_id"])).name.lower(),
        "offensive":r["offensive"],"vulgar":r["vulgar"],"abuse":r["abuse"],"sarcastic":r["sarcastic"],
        "target":tar if tar else ["None"],"language":r["language"],
    })
test_sha=hashlib.sha256(
    json.dumps(test_payload,ensure_ascii=False,sort_keys=True,separators=(",",":")).encode()
).hexdigest()
if test_sha!=EXPECTED_TEST_IDENTITY_SHA256: raise RuntimeError(f"ordered TEST identity SHA drift {test_sha}")

valid_ext={".jpg",".jpeg",".png",".webp",".bmp"}
by_key={}
for p in INPUT_ROOT.rglob("*"):
    try:
        if p.is_file() and p.suffix.casefold() in valid_ext:
            by_key.setdefault(norm_key(p.name),[]).append(p)
    except OSError:
        pass

image_map={}
for key,name in zip(master.record_key,master.image_id):
    cand=by_key.get(key,[])
    exact=[p for p in cand if p.name.casefold()==Path(str(name)).name.casefold()]
    chosen=exact[0] if len(exact)==1 else (cand[0] if len(cand)==1 else None)
    if chosen is None: raise RuntimeError(f"missing/ambiguous image {name}: {[str(x) for x in cand[:5]]}")
    image_map[key]=str(chosen)
master["image_path"]=master.record_key.map(image_map)

h=hashlib.sha256()
req=master.sort_values("image_id")[["image_id","image_path"]].drop_duplicates().to_dict("records")
for i,r in enumerate(req,1):
    name=Path(str(r["image_id"])).name
    h.update(name.encode()+b"\0"+sha256_file(r["image_path"]).encode()+b"\n")
    if i%500==0 or i==len(req): print(f"[IMAGE SHA] {i}/{len(req)}",flush=True)
image_sha=h.hexdigest()
if image_sha!=EXPECTED_IMAGE_BUNDLE_SHA256: raise RuntimeError(f"image-bundle SHA drift {image_sha}")

master["target_json"]=master.target_list.map(lambda x:json.dumps(x,separators=(",",":")))

# Worker manifest: exact TEST identity + canonical labels, but the locked-test worker
# never uses TEST gold labels for tuning. Gold is consumed only by the controller
# after all frozen logits have been emitted.
worker_manifest=master[[
    "official_row_index","record_key","image_id","image_path","split","cleaned_ocr",
    *BINARY_TASKS,"language","target_json"
]].copy().rename(columns={"record_key":"image_key"})
for t in BINARY_TASKS:
    worker_manifest[t]=worker_manifest[t].map(lambda x:1 if str(x)=="Yes" else 0)
for lab in TARGET_LABELS:
    worker_manifest[f"target::{lab}"]=master.target_list.map(lambda x:int(lab in set(x)))

PREPARED_MANIFEST=WORK_ROOT/"prepared_manifest_FINAL_TEST.jsonl"
worker_manifest.to_json(PREPARED_MANIFEST,orient="records",lines=True,force_ascii=False)
PREPARED_PARQUET=WORK_ROOT/"prepared_manifest_FINAL_TEST.parquet"
worker_manifest.to_parquet(PREPARED_PARQUET,index=False)

print("[CANONICAL DATASET + LOCKED TEST IDENTITY PASS]",{
    "rows":9318,"split_counts":counts,"ordered_test_sha256":test_sha,
    "image_bundle_sha256":image_sha,"worker_manifest":str(PREPARED_MANIFEST),
    "TEST_model_forwards":0,
})


In [ ]:
# CELL 5 — MATERIALIZE + STATIC-AUDIT R3 FEATURE-FIRST MiniCPM LOCKED-TEST WORKER
# Pattern aligned to the successful InternVL3.5 / PaliGemma2 / Llama FacetGate locked tests:
# GPU worker => frozen Gated float32 feature/logit shard only.
# CPU controller => frozen SHTE specialist in float32 after feature merge.
import base64, zlib, ast, hashlib
from pathlib import Path

MINICPM_LOCKED_TEST_WORKER_SHA256 = 'd54a4f06f119984eb1d6b0ad5a059355a694980123566336f5cd03fcd1283349'
MINICPM_LOCKED_TEST_WORKER_ZLIB_B64 = 'eNrtfe1S40i24H+eQqXee5GqhMAUTVebUcdSYKrppoAFqmfu9TgUsiWD2rLklmQD5WBf4z7QfbE9H5mplCwD1dOxsT+2Y6aQlZknT2ae7zyZiqezLC+NrHCM34ssdYxpUN45Rh6kYTZ1jLuguEvioWOU8TSCf/NgFA2D0cQx7oM8jdNbaJdHG+M8mxozaAl1jZhBXsLPDfGczqezRyMojHTmQL00hEf43yzklpenZ7LV6TS4jWSzMstHgAz9cdOUAKTVb3c8T0dlnKVBgkUnDKyYJBHg5k6jMo9HhQQcjEZzwP7RL0ZZDkMZd+RTsIhy6NSf5dEoLgCcLLgLplMYop9kBQyzKoaHIEn8MVXzi/kMO+DOYYLSYpzl0yhXPR/Oy+xzFkaJQ4+XeTaKiiLLHeNjXBaHafjxsYyKoywdx7eOcRuV/igr4hQgj+6icJ5E/n1c3vkw4dP5TEx1NC4l+LMsD/TGWOZPuT/AeRYAjvTTB7z8yTAufcAyxsVrNvCLMigjP4xH5cbG0cknD0nCTbIgLCxcTisr3ChdxHmW9s2Tw6PezafDm55/dHF+cvrJHNhuHgWhX0YPpRWloyyELjxzXo63Ppi2vXF1eP6rF6elBsWF/i0TC0zH3IFKB8bZxdHhmb+2blWsWvz94ursuLUylfjXp//Zg8odxOG499vpUc9jCgqjRTyKrLE5modBd1mBfjKNeCzIDMvcuPCDRRAnwTCJLNuIkiIyzNFsbtobUJG6eePtdoFtYii5mqfILr08z3KAfgI8U36CiTVg7b5GKaxKVmajLAHO+WMe51Fh3Gd5EvpF/DXydg+M26w0lgT0iTtgrN3ycRa98Qhbs7Uv8zMs69HlZ6PqMnqIRnPkkaqzoy/Hh8anyy8FANfGWAApiBnh/uyN616PJxZooW/mc+CMKAphoTdYPrj408Ja73DaYC3SmasXaWX/Zu2+fft+d6uDS8bdToN0HiT+ChANKa2KDzynVdvYuPhy4xFVEnbZvJzNSz/PshIQPDAOr248qLFtBnkZj4NRWZgHxtGv/A4YazSZZTA0ent49HOPC8ZRUM6BYUYBVDGRGj9xQZLdFuYGcJARGnFqWADeOfrVoaYO1LK7oTudhHFuIcMBXO8mn0dO9BAXpZ9N6Je9ARVPTs96HvzdHpuw6pMo92HCJksiOxf5LTE3wmhsQIdWtABIztu3k3u7u2HAfyB5vKWJC2528V8X/7Fsx0QYZheBOCa1MrtV4ydqi0LEyGZRagksHDMwnRU+RUk67o7d+zwuI4sEQAjSu7Cgb6hd4OwExSiOvZMA2MABXIN5UnpFmdvvzH+mQFTYG1At8bC30wV6Rxoy+4I6txR1Dkzn2zpwxsm8uBOTibNU3AW73+/74xj4ciYm6c4TWsvlUss+aEzAzNnMh5tiqFSG/+HiDnFxYeC5lQTTIUiFMck068Pbzs7uHv1jO8PNTbt7585nIYzBGtpiaYBwUuPOvYsewvg2KkqLUQzKbBqPfBwodJwJJGdMujP7oJzOvJlLIr6Yj8fxgzVz+eHdpguFm1SF14NlqzZnmROnIayzt/vi1KmV3qSVBrAgLkE/JLAaFvTgzICnEOFxEpRllEJfKWgp6GP4uwOKBBDyTFOgD9zmLZ/kSjPDgpDkJtjC7kKVPjdzc0Agnlmb7qY98KCQ2kUJtASFmoLSSQEF7AY1j11fkYmzwDWBUhcmYFpYBFlOfhPXhTM2l9zr03Ly5KLM11YHWvIYR0GapTGocVDueeBPokdrIjqe4HTBrw2JwYwY3kQ16YIeQfNnmKWRa2qYwlAmLgwkLwtcSaTFiTfpJ0hsdncgAE/UfJuuWBuE6Aqeub8DMq7Dgd4Kob5d/nflhVl1ZbYVDvQJmGwI4QKDhhpoXKBB0ZiA1tkRZXkE1Dm0cvOfLhX2Dz8O/uneR/HtXfk/YDCmM1ES4IWh/BnMK8ntD5NsNCksNDsF5kUpNDuaLFTgTIMZDGIUoAr0NkFlbzqMa+GDqH1kRlEIp6B5NZIsSqZIA6iA1t/kGin0VK19q9I/hx4XkSEVMsxoGCPFGtN5URqjLC3BBAPGmW0lIKgTsugODJjpwlsmoDOgb7vffb8zMOpcolAiGwQtAqz5JCjoDu0vAtKfMPcw7Y6SoCjicRzlsEJhXIzyGCxbQhEkfrMHMp4mNnc0kHODy6Tg22+8zl8yA9EDqObk0QCGAlRB8IIxv4W6IaTBGLTIXaMagGGNItToMdRgVWsbF1dGfVCGpXTMFr8BiwDEQoGicqlG8dQ642oyJ56q2d+pESOsQlPyFGV/Exdxc2C3ld1N4P0dEPF3xlbjPwNmJVh5u3F84s1CtqhJd5CVIwz60AfDKB6DkjEHTgK+QiF04s3V4em5d3zSPz5xi1kSl55nkrVvDlwcfumjvniwwjybcZMD47fDs0aLRZDgUgHLPNPspnd90+yJEHJhzKUPMOZRYZkZaLJRDKIkz+4ZimmvgbkhqIwGART2w973e8h6+A6QhDc/vt+VL7B7ftNtIT+rguPI5o5qBkrh4+n54dV/eDSpQ6Ca/NEvg2JSmAMw+g7PP3FJEqS3c/QNwfIHhsPCm8OrT70bLgbhhg6ULGQJNYSRWwshkeqstXCsYZYlDljJ+Ne37a4gKDSQFnZ7EygDisqCEtshj9xGOT7SO1DodSj0FmD9tFPTfFiEem1hu6yMbXcEInecJSE4NMCIS3CQHKCVeQR/HqMC/4X/z8AVJTFBj+YT2zRAHg9qjIW3FvCBMfWWD9qLbkxiKXYesM8onYOjjHocp9xW9kQhxexUrO1vSExSsMzTSZrdp4ZcHGO5kCwrBjvtFwPGk1aGVRzQnwX/V5Y01VwKGSZoAKxq0h/CjunTWkKj/sPAJsQJbaadgROi+BUah6b9/a7tSIASuwZImjmEWFUAwaBDSjJYUgWGSWwtXmNRodtdPjyZOpJMp88h+QSyiGTDFmpCQ6jFjY/+5cW11x+Qs1NW4xV2awbLPZ9ahMGD3h8yW79E1yuNbr2K/7agyYFBUN1gBuZ3aIGYtj64O6CcH6wO/IUG2/gMNR3wD4E9b1aQ4PG8CgltUkqclIMWhG6+DR+elNoi0CuHPWZPeMwMtl7vpqXehuYe6f4BeJXbLK/JOvEl9/licchBNJ2lIldU0dZX4DpeIzR3WAI4POFg8Fg4AEcRkmrBE6q1uGm0qIRfVhRmF/iOsQDtM8rhlQ/KFGT3IwqFNrUmNf9MBry2ybJb1XSqgleLj7nEtqDxaEKikDUgB6riEBRfDnOKATlPK5DvoBhkGfjeeTTNwHEC3ydil3wOhuY4AFNRmH7s84Au8tlSsKZSrkVR6oGeQjEG0ng08frTgWao07vKCHrw6IU7y0AG6o5BHCKNotEIALto+MTpPFI18K0bhKFF9WotwY8FQyK3HhwT9cwiSaZ+NB1GIfpx6KqnoV4Hik2lCx5qXlTAdmBlZYM81/6CH6B7M/jfowddCsiBcw72mV0rB/QeQVGRpMZSQka9eejyZAgGexTiuSVkdZTNk5DaRA9A7pXBKCy51YFvwy+2Fcdg5Jm2dGnAUMLxWWIgf3ir4VVLDYGqA5vtDcFyIbpQRcN0SK/9P+YBuBkkP810vGe2VEFiCrP5EPwoqr0O1CibzuYYWm1K485+VRtHCeP64BeTeCbcs8Ij25TtFAlNL0f1QRB4iqeeCjev8I/q6K9hJAWORh5/JYMRynGivT8cGmTrgBOwBEHM+NNoChMIEqYBj2UlmLgzb2ma3bb47JMDpJn68XSWRFOQQ+zfmUU4C0x9OqRBsLGBpOFpRHIgCMzTWR+LSDijuSTpH1+6PCzHvIvDECx7DNWaDgYz31QBUiDgaITiUa80kAZHm48kaV1rYIR5PC67xrKt+02t4iYzpQgRr0X4j3kEti0YWuvRraq8CllV/XlUVTUNUa2Ci6xDYVYWxBuavMNqUpwZ1AaeX91uMavawbMrA9/+bQ6rb0mx/53RQ9/TwP2bBL3JYj5CxTOeJ0oGsSPKbmaNzEFqT/HXqHAB0Onny4urm8Pzm65R3kVG00E1wiwqjPOLGxmDp1piM+A6vj07vdz+DXors3vwb8sMNcujcXLZ2UfgX8BLymsbS563534PdorxzgBpUIDgHaKIA2PC3dt3O46xtwXv0cVeQG3Edgrw4nSUzEOwpS8/A1R0GoO8cA3jBnApRjHgCTiPAEU042lbDYX5+YXBcsA4y64OjW18QfIEt0IQFo4xmEYgjWkqEBqZk0BdCTh+pQHzmWypzTN4D2jgU2HA5KDMc0Hska+PGgS6D5CpcxcZ3M3mJUiw7HeXfMMXJeEbr29qEEG5rYFnDl5B6NU6fkw/GtjVFnelCB+R2WxFZpPCCGX+KPxA3qvTlwujz9CUdRXFN2AJLXiDO5tntDwI1bE3YMqiWQnkin9wDqFl9Az+tV5Ez+MATBaY4jEslCJvFIcwiuhNjsgCr/h7ZFd3yGbwHVSzQkwSJ6Ww0KEcodUMHE0dOQzb1q0XBiD4kuNWW50NuRLKzGj+lc3UCposRxwTV1L9UCsqbBSx9N46cChVZFtFJtxUrdYwj0PQSmLkaIEU8yEP29IoarLgnlnVoLmMv4frGiJhcmwzCR6RXXbcIkrGPhHmHxLUty72KrEKORWFWxgBNQTJJlk2mc8EHVSLDuskZ5DYHS05dNC1uLworlYXK6DBpsL+MhLhU2BWtRBR4dfolDVsyrFCsDwwVCh7ERukhGob+lVkVBZJJSkFTftARenrxykavDhMNUpd2K0Ti8Z9wCY0hSvzRRQy7o3QNJNohes3zbEg3Nrsnp/sGZXUMYYRiICIhX6c/h6RQhAzT3PLCMiZbWBX8cI3YagiOtT0X0KvwkCi2C6OUOcE6aM1q5sJ1caPJv4qTQce8surjcqRhjBUyr4V7WoGmyhWok/RYr2Kq2rgvlQJ9nacjrH6y9hdyZZGzuUUks/RIprGRUEOJuLFW/+yss+9hPGUckXmoHQ9zzy6uLrqHd34v/rXPx9e9vyOKeI0sPIbtM/3Gii/nl/8/dz/+OWTgrMj4DQH62pjFVYgMv0zFPZFxAxXeMEosnk+Imce8JAqfT2mrNdxc36D44ag/Gv+zzCbpyGGk3HwwheZ3gW+lEXCI8KQaUNOulQivDlmL5+Yx+Mfrs8/fR+eUBX7vqM0z5wzmIDeRZuKAda2I4WPhot0NelFVTbLwIJ79MyTq4v/7J375xf+b6fXpxfn/tnF1aHv7308vfEPz84u/t47FgNtzhuvEMa6refWEMZcCyExgwzBMQMyNKo4EVnTnKTkoQA/EAY2pTDV3+B+UP0N7eXQdhLu6FCZom+VJUPBEaLGq96X657f+8fp9c3p+ScfE52OFVlXeDgaBo7Wt9Pe6+r2JW/rcOWqFHNsZPgINI3eI6c2FWU0W/HoZLUoYceOKg3sFzbqjpruCjZTJu5K15tYvGk/GW88wS1t3W6qbbS4nqWxdrxvvPYin1tLi31tHKk5igqCAdKEx9MugOX+O3J1W//wPA18oNwsQXXsNNa0daWdMpv5tLXL+7Hku+hzyXuN3PF3II1HGThrvB1qcKxWWm5xipk8Wf5ocIphzcOkkfDicNAyHyl2xn24KLSWa3b7qw1ijYZxuUze2T80saw1F+CJ8aZwIkqUguKkKUFMHR/bkfvdcBi4FY+u8PqDA6FpMNAvI5UFRUqrUVSrrnaQoanoSnbESNAOvucVpCvdKA058WDT3XzHxlxRf5mKzW1t87/qoytwkyHMogp/kq4Ro1DFPNdVewdJI4lSu79T8bKA2X0dH9La1mgBnMYh7mHjuCtVxg468KoA3+/uVrvYcrYFgrhO4pVt61v72pQDK+5+/15utMra9PLVokQF/7WhVIQshAvrXkWvyyYeT4ZkOl8OY6mj9GTIGBbihs6TsgIqIqOfQZJk9wAmiYKxtzSFt+WYE/mwkA+ZfLjFVFjxPJ/JpxCsCH5+UiSbsqPcRvHVfCWph6m091GOsUfCI3Vz2jlHYnQ6dn+rU9Ejx2f8JJ5EXs3HVsEBzkpJlE1M3DptUfay1yata12s7kqgGEALGYEmKXUl4sH8ZrXFd8YJG7ly6aX+9oz3+8b/uo/S95WBzx6w8d//ZfxgUK4EyuHtz2eXMjBlvFv1V9wadivuNGGWkkWP04u/9GWvb2802DfVuZvFSIuj3xAsptsSC3hNN6/lyW9hPTnrRTydJ0AsUTYvahJkBIZpqdR6nY1AlUMPBwaPxluKEiVIBMWI123iyzzPjKYbt40zAuwJs6YhUiimxpgTZ9Qrbn2o9nJlbej7oZZHxo4bUiR5ZbTsD00KfZBUP0Dk69084yf8psU6AZMhhduN4aOmcRuSTVA5TGm9EzV3wpJ/aYhrCe7hBYIbyLSZekdVhtaz5MKN6qpG+Kx6atZKv8LXrvcpXSNzCtBH4EGQySBmSDhEImeGqNHTqdAR84dtRP5q5U6IXr2WcWKX0MtofOtVRyGs3KONJigIktld4L3f5V+YbYTJqzvuzvfOMA4KzwT7JlJ4SV0koDuYFMRbgBwjlNmfXv3kBG89SETsg9a9JW5ZET2G0ZUp48xYh4jkUtYjesQBqaQZpUBZZ6XPZ1m2sUsqflf8onRTnWEQxZfZRW0IFEYI2KJCFpHmy97JTWVoa0yCLZriBeMwbUygxLpJE6y9aIvavDyTr4iOrMbK2CVlbpEhOh6YjNCt8ySvTz9/Obs5PO9dfLn2wZf+7F8fXR3eHP1cj26gPsNBxHX3Azknosm11pRO7oFWwTIycWcsjzArI8ADAMRDwlLX+mBg4HjTcQmePpo9mBltK52mmkSApZ0oQusMsxvFFIZRUXrLFhchtbszfTUqEGsoW1/aJ+nJtIKe2N2FnhkuUZLp4Q1oE4Y2RQ0cesunAwOmKeS8DmXBhRNGE8ej0sxrjgcIbquYOCV7Tfgk+ta7LSaeF06IMss5UI9V4hGEWWR7Hv+eid+tfkfRfXVHUKPSCuGE2DvUXhUT2/5WNFC0EgKkOVrZvXI2aT7RuBP7SojrMpw8AYf3rQen3q1I0OKRUBf97vcD5aHQYsM4PS7aGRzMXEyJdUfZ7NEHKGVmifSpmTizJbLaZiJkZh/gmlISDYz8gNe6H04GXjHR7SkugPHhD1xru9WUOU0xXSMB6mSuxy3RLI8aOYYEC/m+5j1XCTFdEVNA3zuYeU2mkvJJDyDZB6Q9la3hizJqhgAw+iCSimkMErrdnoElE6rFPnYOii9aTb/aIIdB2jVU9TAMZngIBpyNz6QQBTeYpik2zaM8DhLcZMLZicN5kGwF3Ahte+jIyMaNUEXbproLEIUkGRs+CKW49H0Lt8Wc0MPTNs4d2L77DiUJux2NKYv5DNWVqxppxATN3TTLpx46S+gcnMMPK2zUGI86nvKmrNC5Wynf1crvnGZ7aUpAnWN+pFxmW40HqB5Ec8jDedBwf/QUitaD3XwNeMmEKX574t5Gybz+TsegpQRwf119mS327lHSAS3PqaCcKySZNjI4bSOtPAKvsCjzOVkAdN4KSaBuMYuEfCFHKHDzDBVoCS+SU1DYszX2HDUcHBP91JdMWB/CL8GUIRjaNehAdECDxKrRi9a17WiEoGF0TAWfemdfLLtBHXhS6/9KR6BVgRPLRx8DFi3dMLDdt8cCyHV8O83icAUOyOocFOMzeB7r2B3XcHI0JgA2bc4FGtFCPBSeoqZjTEVdPnTbJM+xg2wPoCpvqf9WJLlWqdMqrXXw1OhS5PRjULalR30YnZXM7iYstb9CiQV62/crAyUnYqXeD80140CdL9LuoOqltIcs3m3+GuVZgTNeM3D140orwJDi/hwoTVhxgJhJT+dAJ8aD+aBrUFMjafOTOEiu3CoG2tVS+cYG1sSYWBSkIEWnjQjJ/b03RQ1/J3S5O08LIL/oa2Rtdeopp0JWWXdv7+9tlxJV7O37e/HkgviaznzM5dbaLbx2xrdqWDWHqBFw6bWxc1tzNS31JN7mHJFdxjBFSX1Cwjy4FwmTeJTaAuYS1s9CWD8ribjY5G9upw6HR794u/OuheS0Wd6pg6PwFwPcXQVYemUTIA67HdyttyqgBE2OgtLqL5xyQFtYVaoy2ea3bxfvrM7Wrf22VCW/eJqQAhvRoqoaJ7D00it1ugPrl6oGyx9wASbdqcVVxBYIh29XJZW0uJ+0rOHbGIPQk+6KkOlPBpboAh6BPCsirjZahHxpwNPPgnirEkeB1Y+MrICQgtBrCqKquayyov0ZhrM0F2Z3ATLV7JaO+YvZ/YWj4Gb39ok9Q7RYyOyjPU51YpSSJejQLaeZ1w/j4RHX6vze88dmCYh2dlb8bpwEpWPVSRwUUaFw+M64rpIYA6CxGAMR8zxCmxyAgH9gHHIbTlAss4TOHgE/Yg4RGK7kn26LXTdxdLl2WGjypDnIOEhhy3OTpnNMDuzqjD3rvKpNhSi+TWuHOKEm783RFiOHcSYcxkmjUvxWhzUFhBdixidca0v6IoY04aV5z7mu2mYTW3tkuoGrJ3rhOM6B2owhMtkGMxb+v2vaVaSA58mtbhmxpLOGjnnllzf3Bb8zVrPctBOhsMAyVrm/Zwj/yDWu9Uh4dSOGdveDAD7KI3HgFKhlXnB6bgE6FMBtTaLH+nS4jV0zcCf3914ZnOfp01BnZFti8wiY4vL7e1VYno+pQkntnKoIYISLFyMYg/qZkAlvkTAfhZOGYhbhh2LRiD+sKgUOSHheQPSohyZwlzWoxyKKRSMYEcoXq4AV6nL/xCKENIUxT+M/vOUDKISuFlenNk/N4AbWrZ9bpsBQfZsX5sFp4OXQtFPrfndvYNsHq2d4EC0vxcsZ6PIIrOyKE7C2hi6uDsclFhTVoCgGdMSmj6pWC2DUYhvNjeRv3Y4V3hoHNETw5kBC2w6mw/h2Duzi1beSD+RObQsh7u3omR61KMu+Or1Luvqb+USiSakRuGsoGESKDK2zJ47qLVVn2tbw/p7AT4gfkt6aDMJFcTjpXZxDXg3zbGjOmKeftVifXtx+ZGRv58d9IAhUq6HX4mq3+LuoOGwkF+3woFa0GnMizaRiTpqeol5rYNcEnOoKvDX0JDtZE3pCJRWFW3jc2hCX6hh8340hM+RWI1HfGVfvgYtx07ZgIVwdyeBp+u3M2DYuQWZ9iqbTYBd+nIHVH9S6m+GGcg667zuC8enyi+w5moKZQy85z0ZkZUpqExcEkJQvttkoIltBHmwQ9StdcsM7ZgVQGiqIggKAgThPAmod+gT9eQQYyPzl97sOAJti7huGTyXBh5gplM3wKNMWHW7RD5is21t482yW2rOJyDRddIQfL0aLZqU0iqpRogLcwouXjLZOdIxBjkxjmCVgM5U2q7Zf9EB/t7lhI5I3VSsi0Fc1aaM5GViCiQUDTji0q1RGPimIFOuhOqTfvMHmoTpHqXPe6vn8B9IOqnafAD+yyf9IG9GD1lakY1Qz1jirbdt75FtylFHapXYLW7crH+oGpToTSoMv83k6QvGXjXIL3TdlQuNRyS2ReI7kyWzCwn9LWPicFVrlor18foq5j+0mClJg8is+kPlq1m5fwddyaCIoWGYTTx0RhtWYRClIx/zASEBClVX24zR4wBH5VKOQbk6UjsCbnlB/DmhRXzCrqKZ2mrFHPxuPwQAtfLG7oamDOBQ5fACvb/LmGbyjq8awlVbIQCQMLZMTxSa0sf/mEeZq8QEzdeDYByOksTlFZloOrl9kUTtnB53mSpNiCzp2Af32J1udQb8zqEUgoMLfvB2cbHiSB5Gxo/bDyFhC9gb8xdHNwekHFPtd+L2yYwSz6NLNUnTaYu0E22rQzdiFXHTdDQRI7TszF0dXknrRMQnjsHZUokavwoMzW2NIeBEU3xgEnXp41r+PU/iAU1fZkTSjXcJ7IFNxqcpPOwOcTvJRiYEG1XKJOeYFw2gU9eGoqnZz/f5fm+BxkCQovv/amV7XGwGSskXrcaTOhRfwohg/Sr1EozaG8xBjGWLHWtgp1a4Uhuqms9JCAscrgrpiExgFEecZiyYy5ocFJk/+tLgFX2lp5lkS4bUHBd2aJCCZXa5LV929Ey+fuCH32Sar0LtIHv3RXVCChgEbGiNhlbkPHTqyrhBIWtgq9G+jFEMVIoKJfVBKvT4OAt1aVx0QZ1lBm/8lGB8TFHDtcBq1GkfM0fwkyC65rRZPyPpdY6mWqy1DbsFer7dsA6a5FLViuZ7f0BkTmGjY2qcE2rhFhus0A19kvRZWHtyvtx6oEMO48H83BA9hdIfX4uDNGkATpClqPfWF8NmBiVYyZlDJIQAjLhfSdthEXpE/Bis94YsBGWOOY8t7BvG6ubssCYHuK8Jkqq9+00nFBA/di5KqlUVvXHJIdhydVFDfcqN0PgVNqF0S0Vy29Ry3biVN8OGR+dGTnkdgmWojV+YFDZ2EW9vKqcv21vfuaAN1OkKYEMH4gijEFSzQtVVNtCMKHTWDaAtU9FDVfC3ZgrMapOE6al0LT1LsaMfT6vC9WrLOwajjjXbeUXqHeKUMoxZRpQ+zRdnQTQ2vsZZ2PSZqTZ02TafBgYDgtdpNyvYFYG9WodXmftBKQ/X9aBrYFtmdp8cFqLTxGIQQ2bErV9UYFZo8IjENZJqtxUQX+Emipg5pQ91bpB11+N2xyjh0YNx2/TYsvJ4HR+2I+bE1ayFwhtLaU/ICnxs23/CngM82/m3UoYfhT6MdesAG0K2trjJkLLuAnAyC/V4zyaGgdXbTTB1XKSbvUJcrXhHZf0IYo06px1Ab7JoVgvPIzmnc1EVpBHQtuEuXt9LlWUJMBeUdXnoVFAYwdpdeevDk8tUHeLX0p4+iL/AMvJrjA3D4KNUoiYIUTAZ4ifdXivge3hQoLnkrJn4syIguRHkHNXWF/6L1IQ7LgM9fcVyl/skl6nOTAQtvIJM+PQwGktlk7MUEje5okUJdBnvPimd8AVLgtrxbqcfUzoVQs67sTbRYspwJWd2AScNBI5ge+vVKA3nEuvtcFKLitgwMV2TRGpAKAV5t0ryr/Zuz+CFKxH2D9eJv7F3rZrsGVAk1vLbGEwPWBBkw4rNqvvL9EID9U+Wzrl2DdvNcMBTTinEXl0gALKsM0VIQG1kp3orZInHXJnSgH1DC2eOqr5w7pk7a89Hnz9TTTn11q7ZK+hZ41Ga5hghVIkRAabaBeSGSLGaKUNriJRXNt3izZRoXFIWTU7XUenySIUt43da3VLvDIPT6FkhkZ7gVsEKAH0R+BIgE8d+8gNhkKxAnHNvu3VHKDkA+a9gu4mIONk8B5kJ1YBvj+BhgXUJrTK1UO0NKlXvfYDiqOE1l+3jfYBNJKtfPr66yDO+t4IqvMwArZVBdXc+hvTJLWzQ29SEGjm/pXg2dYdU5I0rSaOVacoc0EGP0xUQyxj1uD1r/uwYaD/fVWR1WwNeaktUys6vrBKoyOTdivkRNnI8dnRo0aO2EUc2AGqDBHVTqmK9XoxsmFhHtfYOdshWHWztdIhqtk363U+3osM6iXRJ6tA/WzKWYI5D4BR0oa1nx2k2UONF6FxTCFF3g5D63ZtoiVRk7npbrxBggmDV9rjBqtwLUD7rDAZmyFR98cweqVb92FWSdqep3Qq7eAdqGhfHvHi5zWvc88eBENQLj36v67WqE6m6TJYyVCgM32JJgZtaYtoJIGVb2GkH8dq1QW9/3FuE5CvIQ5UdcPipxXEdBjUNhIMNo1TS2D5EtU+lHYYvVPoXYrO9etpozLKda1VBb53rjbSG427WP6FucU8TEIu5/MqhCvzoXOU2OkDf0wm8ur6lUp6bpHbO8LWlrEUuaBtuYe6P+n8Q+YnDrLZWeNbXgepezc1b9HhmqHOXPhCor+1XgJ6G2kh2iCguu12nSxQpA1I8CS6HTHbNt+cxu21sNXEG5sKOoNnCp4rR6lTdKUtSvlqk5Y1zBrvWhu0Zd8/Pp+SnoNv+3ve/988Ob0996/tHPhzf+Te/z5Rl+cOfy8Kp3fuOffj781PM/H1792rvSfAJT8EeWiBs8AGTvH4dHAODw+lf/9Pz65urL0Q1e5HF59uXaPzrrHZ73jn3w4PyL87P/0EHhkjeAXF98uToCHK56J6f/8C9OTq57N/7Hw6Nf4RHvCDnuHV0c9/yr3t+vTm96prxrWPP3BLVr+Za1/MEqkbL+GsmxFu6V+fVgIjyfiSquukQuKtoTWWGJidobHqg0Ytcc5FPuqRDD8zIbBUUpTquIm0Ppcz1Oyz2Yeh4cocZXnBaOj5LGXb341NLHUXn7ugDw9Crs3tbkQy1GoJesXOWqF3J+yhuBPjDg/l631mutMiov7TTGHzQadVrbqu9PSMlGCf3OWqBObYa8+nzVINblY20+Vs0Jp3GZpHS2Q5kLUsGuBsQpGt4fbgJrre76xHyS5qkuGRce46EEmdthY2zIklrrRFwpdokJaSqwkm5xExHhxDy2mNJoReecgacv522S3RKHeLyH3pbO/U2Z3M0XtlIJ8tsrgmdqoRyFxsbG/xSfacto2158DEdwrjwSPAowgdaqUjgx6Q5c2ng6n27dRhl+w+3R4FpGNuT70Ojk9Ks2nvO5yNa7w0t+6W5w/ldoiUn0CJ4IrCJ/RiOnK+y12+o9Dyre+jud/f0fzfqHMOJngrmNG4wF+uICPhHSlUCkpX0fFP7QU1kTcvYP8L1cU/VSVYvwAnT7gIv5B0HzndBrCEnozI2TbASmjKOl/qssqWVNvb//sOc0dPP+3k5DE++933Hq+razs07NYsmKLu10dj48KU8ajB8rlN8eqaczSCz1M5BoB4jq2j1Ii+orAmtd6cb6yHsqxDpJ04w9o6cDRXfeMtS2pWjBuvUFs2qlt11t2SztiqFWJhDH4xVh6qTnvH0Lk0MDxtzyyjSU84I3sLemxJxdHP3aO+ZEH9TtXcqBqic+ibwklaaFufEtaTMqOsQ7hqxcSJX7nE7k0+c/BHE9c8b6TE8+mpd3WS4TjcdJcCuTEWF4RRyqpDFx+W912bFCQXke2D1q7kLei4XfBnkVFtBI2OV1fCjf0FWnu7k/6ibGj27hmQZxORX0Zn4/3g9GHz50oh93dvfD/Q+7493xXifc/z4KO+OdD1Hnw34n/DD6MN55vzMMop298McP4x+HP4bD/b0O/Gu+DlnZ9TpcMYsu2gKHLh4/arluK5cSo5KXqV5V/ppkBjH7EehGTGWYyBQ4sRx0ozFHFTAnLTKuf77pHRgxAxDJexVUvIBrTHnO+MmABO/dA1N4EaWY0UQJbq+6IgzzoV91SdjzF4TVbnWu7gYTQ9ZTCUXmElPACorFHd6H3opgo+gV6InLc3Ae268uexa9jaZKMGo6YY3a3diQUjqMc+1bitpbSnDSfr/iI4eUZ0KfKKJsC63x9ljdDIJX95d0xRKzrfoGon99c3h10zvmT13gtyFuc1Dufxba5dXFp6ve9bUAh8tYQ88l5FGl0BmWWmeqbO2qVfn05hX0aXCn9MVDzHzWlop63ZYdyAwI7sA1zMrjwuv7Qf/QTdwzPpfASbSsx4k3i+puGtcUmzX6Z0TqQ3TYhTf5Hjv0L99fjMcRODRCEZKjqZJa/UXH3/E7PqsOHzuXa+JfvfdPeoc3X656uqdo0ldBu/htzvqXKOmzpmaXPmUqqqqvDnSbHyFofneg2/4xgg3pbtMnVCVjdcXXsepvVe114qL7gjSR42vn5u6zrC7atqoLbtlaJNvhEZFb3JYkjdbd2xcfijBpQQQ1FPgt4izH5OzujkRWiV1fJQzDoh9dfvFPzi4Ob97v+ocnN70rtZKfe1efpJ9O3xYF8pnO6h8Y3QALIw4fRJoj57bRGqtvajm0yra9gfjiLrfmGCsfoKvdmRbX97wRun5TZXbvIVy2WeOB5uis2rYYAGicjG1eyl+gHV7Q5VNAJeF8xKd8alnMMkJH+dcyS93VQC62y+1ftr9+opvr6WxCSN8QFQaUuFwS4ZIY1xQgcAgpxamrj1BtuC9rLmzbN9PI+qOt75ZCPcNLC4Ihl2GqrbZjjpzWrMxbSmY37JsLU9sSqUIvHLFQuyT0NW/4Kxwn+Sky7etXWiCK4JZ/LdzfSd0i4F/+WsBffflhJagTFEGeB49Wn4HdJv1S66zeA/ollrjopFz9RthzPVYfCoMetCOe67qSgwE9glto1vtXjUscCCWrgHuSp0Ff288Pz/fzVAstWem7jv1vu997nHdMPz2PY6c1PueQjfaF3LoKdpYrh8L+FT0mrYE1iqzezRqlFuJ1YV0YEHjDWYmMo4bVAmWdtMb2q7X/pLKow1gjwfVKT/VUXfJKNSdOOe9yNUwHR+0h0jTmaiWd2gCxBliVIkiCEg4cI1Bdz1whptlNrYeG2BslpbJUQOlkJMA1hboRFyniM9+7Sh9vNh66D2vkJXpJ1+CKgDVFHLpNNF7tqAtRjh9dRosLRXqcgxWC31PBD5+soujSKZac0jUBjUrTTRyGjTs5lhS2jkWX29i2Y7GU1F+weNPfNPjXsX6Q76urMKz38p2QYY6FrbVrkwNPk2o5RhJq9xTQGZRAHHp849HfOqO+tIBdfUpm4rwefmLVW06e8MJAbynga8fzlvyiHpoOZMi5kjV/FhUC1I4KFWmoiJ7MlatS8cObMoobyABuCz4t0dwGVnpsVyAokEIvDm+ET2dfv9XLcaGNuQFIFsEi+kofG0OulZ/8ElBZBKyyg04V/bydYYwacWtqjTYDhHSRJkYTYGV6tIMxv3R2P5gCCDMIgqAvyBEAwTTN5usVEl+FooMgJns9AGLBOgTmyteDkHxYh6K481sASSZvgqqMhW8ApkuSJsCalHkdUHtjCvaD96/7mHSsUGnl658Pr9Df/P8u5ssuprp4X973xgvI1/93X/o6wFog1GsDRvVONfyzDi4V8txL84FNKdLwolL9UC5+hwO3EnbFRyqqQBXSJVCdFOB/iSusG0Qy1opEWO0LmiBV1Yj18KCQuHIU4P/KvZt684yhl3doZGdJ6I8A22EuEgFaaupEQAu2UnE0SkABJFS2iApxDaickFpgqBX1ehW7YgbdLG9t2aiDQYNaVOpblRp/6xali7hVuM1CJdvLl/c1mg6ZgxUVoTKVn1qhBalWzEPIfX0JBw1jVoGxN/4PtGuM4w=='
WORKER_SOURCE = zlib.decompress(base64.b64decode(MINICPM_LOCKED_TEST_WORKER_ZLIB_B64)).decode("utf-8")

if hashlib.sha256(WORKER_SOURCE.encode("utf-8")).hexdigest()!=MINICPM_LOCKED_TEST_WORKER_SHA256:
    raise RuntimeError("Embedded R3 worker SHA mismatch.")
compile(WORKER_SOURCE,"<R3_locked_test_feature_worker>","exec")

for forbidden in [
    "import torch.distributed",
    "dist.init_process_group","dist.barrier","dist.all_reduce","dist.all_gather",
    "class FacetGateTarget","shte(","z_target_facetgate",
]:
    if forbidden in WORKER_SOURCE:
        raise RuntimeError(f"Forbidden R3 GPU-worker token: {forbidden}")

required = [
    'AutoProcessor.from_pretrained(CFG["model_id"],revision=CFG["model_revision"]',
    'CFG["model_id"],revision=CFG["model_revision"]',
    'llm_int8_skip_modules=list(CFG["bnb_4bit_skip_modules"])',
    'len(src_modules)!=253',
    'images=[[image]]',
    'nat.get_vllm_embedding(model_inputs)',
    'nat.llm.model(',
    'native_binding_canary()',
    'd["v"].detach().squeeze(0).float().cpu().numpy().astype(np.float32)',
    'd["t"].detach().squeeze(0).float().cpu().numpy().astype(np.float32)',
    'd["J"].detach().squeeze(0).float().cpu().numpy().astype(np.float32)',
    '"feature_dtype":"float32"',
    'minicpm_facetgate_test_rank{RANK}_STARTED.json',
    'minicpm_facetgate_test_rank{RANK}_PROGRESS.json',
]
for token in required:
    if token not in WORKER_SOURCE:
        raise RuntimeError(f"Missing frozen R3 feature-worker contract token: {token}")

# The Target specialist source is frozen in CELL 1 and executes on CPU in float32,
# exactly as in the successful InternVL/PaliGemma/Llama final controllers.
if hashlib.sha256(_common_source.encode("utf-8")).hexdigest()!=COMMON_FACETGATE_TARGET_SOURCE_SHA256:
    raise RuntimeError("Frozen FacetGate Target specialist source drift.")

# Data-independent float32 specialist preflight BEFORE TEST is opened.
_pre=FacetGateTarget().cpu().float().eval()
with torch.no_grad():
    _v=torch.zeros(2,1024,dtype=torch.float32)
    _t=torch.zeros(2,1024,dtype=torch.float32)
    _J=torch.zeros(2,1024,dtype=torch.float32)
    _z=torch.zeros(2,7,dtype=torch.float32)
    _zs,_aux=_pre(_v,_t,_J,_z)
if _zs.dtype!=torch.float32 or tuple(_zs.shape)!=(2,7) or not torch.isfinite(_zs).all():
    raise RuntimeError("R3 CPU-float32 specialist preflight failed.")
del _pre,_v,_t,_J,_z,_zs,_aux

# Frozen-checkpoint FP32 smoke for ALL FIVE seeds BEFORE opening TEST.
_checkpoint_smoke = {}
for _seed in PUBLICATION_SEEDS:
    _bundle = BUNDLES[_seed]
    _cp = Path(_bundle["shte_checkpoint_path"])
    if sha256_file(_cp) != _bundle["shte_checkpoint_sha256"]:
        raise RuntimeError(f"Seed {_seed}: SHTE checkpoint SHA drift during FP32 preflight")
    _state = torch.load(_cp,map_location="cpu",weights_only=False)
    if int(_state.get("step",-1)) != int(_bundle["shte_step"]):
        raise RuntimeError(f"Seed {_seed}: SHTE step drift during FP32 preflight")
    if str(_state.get("protocol_sha256","")) != PROTOCOL_SHA256:
        raise RuntimeError(f"Seed {_seed}: SHTE protocol drift during FP32 preflight")
    if not isinstance(_state.get("state_dict"),dict):
        raise RuntimeError(f"Seed {_seed}: SHTE checkpoint missing state_dict during FP32 preflight")
    _m = FacetGateTarget().cpu().float().eval()
    _m.load_state_dict(_state["state_dict"],strict=True)
    if any(p.dtype != torch.float32 for p in _m.parameters()):
        raise RuntimeError(f"Seed {_seed}: loaded SHTE parameter dtype drift; expected float32")
    with torch.no_grad():
        _xv=torch.zeros(2,1024,dtype=torch.float32)
        _xt=torch.zeros(2,1024,dtype=torch.float32)
        _xj=torch.zeros(2,1024,dtype=torch.float32)
        _xz=torch.zeros(2,7,dtype=torch.float32)
        _out,_diag=_m(_xv,_xt,_xj,_xz)
    if _out.dtype != torch.float32 or tuple(_out.shape)!=(2,7) or not torch.isfinite(_out).all():
        raise RuntimeError(f"Seed {_seed}: frozen SHTE CPU-float32 checkpoint smoke failed")
    _checkpoint_smoke[int(_seed)] = {
        "step":int(_bundle["shte_step"]),
        "sha256":_bundle["shte_checkpoint_sha256"],
        "output_dtype":str(_out.dtype),
        "output_shape":list(_out.shape),
    }
    del _state,_m,_xv,_xt,_xj,_xz,_out,_diag

print("[ALL FIVE FROZEN SHTE CHECKPOINTS — CPU FP32 PREFLIGHT PASS]",_checkpoint_smoke,flush=True)

WORKER_PATH=WORK_ROOT/"minicpm_v45_facetgate_locked_test_feature_worker_R3.py"
WORKER_PATH.write_text(WORKER_SOURCE,encoding="utf-8")
if sha256_file(WORKER_PATH)!=MINICPM_LOCKED_TEST_WORKER_SHA256:
    raise RuntimeError("Materialized R3 worker SHA mismatch.")

print("[R3 LOCKED-TEST FEATURE WORKER STATIC QA PASS]",{
    "worker_sha256":MINICPM_LOCKED_TEST_WORKER_SHA256,
    "controller_revision":CONTROLLER_REVISION,
    "model_loading":"exact model_id + exact revision (successful development path)",
    "gpu_worker_role":"GATED_FEATURES_AND_LOGITS_ONLY",
    "specialist_execution":"CPU_FLOAT32_AFTER_FEATURE_MERGE",
    "development_dtype_boundary":"v/t/J/zG float32",
    "all_five_frozen_shte_checkpoints_cpu_fp32_preflight":True,
    "nccl_process_group":False,
    "distributed_collectives":False,
    "lora_targets":253,
    "common_target_source_sha256":COMMON_FACETGATE_TARGET_SOURCE_SHA256,
    "durable_test_start_progress_markers":True,
    "TEST_forwards":0,
})


In [ ]:

# CELL 6 — PREFETCH THE EXACT PINNED MiniCPM-V-4.5 REVISION ONCE
# TEST remains unopened. Workers will use the same HF cache.
import os
from pathlib import Path
from huggingface_hub import snapshot_download

HF_CACHE=TEMP_ROOT/"hf_cache"
HF_CACHE.mkdir(parents=True,exist_ok=True)
os.environ["HF_HOME"]=str(HF_CACHE)

token=None
try:
    from kaggle_secrets import UserSecretsClient
    token=UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    token=os.environ.get("HF_TOKEN")

snap=snapshot_download(
    repo_id=MODEL_ID,
    revision=MODEL_REVISION,
    cache_dir=str(HF_CACHE),
    token=token,
)
snap=Path(snap)
if not (snap/"config.json").is_file():
    raise RuntimeError("Pinned MiniCPM snapshot lacks config.json")
if not ((snap/"model.safetensors.index.json").is_file() or (snap/"model.safetensors").is_file()):
    raise RuntimeError("Pinned MiniCPM snapshot lacks model weights")

print("[PINNED MiniCPM SNAPSHOT PASS]",{
    "model_id":MODEL_ID,
    "revision":MODEL_REVISION,
    "snapshot":str(snap),
    "TEST_model_forwards":0,
})


In [ ]:

# CELL 7 — R3 FINAL LOCKED TEST: FEATURE-FIRST FULL FACETGATE, ALL FIVE SEEDS
# This is the ONLY cell that permits MiniCPM TEST model forwards.
#
# Full FacetGate:
#   internal Gated route -> Offensive / Vulgar / Abuse / Sarcastic / Language
#   internal SHTE Target specialist -> final Target
#
# No training, TEST threshold tuning, TEST checkpoint selection, TEST repair, or TEST rerun.

import os, sys, subprocess, time, json, gc, math
from pathlib import Path
import numpy as np, pandas as pd, torch
from sklearn.metrics import (
    accuracy_score, f1_score, precision_recall_fscore_support,
    average_precision_score, hamming_loss, confusion_matrix
)

if not AUTHORIZE_LOCKED_TEST or LOCKED_TEST_ACK!="I_UNDERSTAND_THIS_IS_THE_FINAL_LOCKED_TEST":
    raise RuntimeError("Final TEST authorization missing.")
if not all(FIVE_SEED_GATE.values()):
    raise RuntimeError(f"Five-seed development gate is not frozen PASS: {FIVE_SEED_GATE}")
if not torch.cuda.is_available() or torch.cuda.device_count()!=2:
    raise RuntimeError(
        f"MiniCPM-V-4.5 FacetGate final TEST requires Kaggle T4x2; "
        f"cuda={torch.cuda.is_available()} count={torch.cuda.device_count()}"
    )

# Audit mounted previous locked-test outputs.
# A completed/progress-bearing prior TEST remains fail-closed.
# The exact known R2 dtype abort may be acknowledged only when it has STARTED markers
# but no durable progress/shard/metrics, because it produced no usable prediction or TEST-derived decision.
prior_started=[]; prior_progress=[]; prior_shards=[]; prior_diags=[]
for pat,bucket in [
    ("minicpm_facetgate_test_rank*_STARTED.json",prior_started),
    ("minicpm_facetgate_test_rank*_PROGRESS.json",prior_progress),
    ("minicpm_facetgate_test_rank*.npz",prior_shards),
    ("LOCKED_TEST_FAILURE_DIAGNOSTIC.json",prior_diags),
]:
    try: bucket.extend(INPUT_ROOT.rglob(pat))
    except Exception: pass

known_r2_abort=False
if prior_started or prior_progress or prior_shards:
    if prior_progress or prior_shards:
        raise RuntimeError(
            "Mounted prior MiniCPM locked-TEST progress/shard state exists. R3 refuses to repeat those TEST rows automatically. "
            + json.dumps({"progress":[str(x) for x in prior_progress[:20]],"shards":[str(x) for x in prior_shards[:20]]},indent=2)
        )
    for dp in prior_diags:
        try:
            d=load_json(dp)
            if d.get("worker_sha256")==PRIOR_R2_WORKER_SHA256 and int(d.get("seed",-1))==42:
                known_r2_abort=True
        except Exception:
            pass
    if not known_r2_abort:
        raise RuntimeError(
            "Mounted STARTED-only MiniCPM locked-TEST state was found but is not cryptographically attributable "
            "to the known R2 dtype abort. Fail closed."
        )
    atomic_json(WORK_ROOT/"PRIOR_R2_TECHNICAL_ABORT_AUDIT.json",{
        "recognized":True,
        "r2_worker_sha256":PRIOR_R2_WORKER_SHA256,
        "reason":PRIOR_R2_ABORT_REASON,
        "usable_test_predictions":0,
        "test_derived_tuning":False,
        "physical_test_forward_upper_bound":PRIOR_R2_PHYSICAL_TEST_FORWARD_UPPER_BOUND,
        "started_markers":[str(x) for x in prior_started],
        "failure_diagnostics":[str(x) for x in prior_diags],
        "retry_reason":"implementation-only dtype repair; no model/checkpoint/threshold/scientific-rule change",
    })
    print("[KNOWN R2 TECHNICAL ABORT AUDIT PASS] No usable TEST prediction/metric existed; R3 official evaluation may proceed.")
else:
    atomic_json(WORK_ROOT/"PRIOR_R2_TECHNICAL_ABORT_AUDIT.json",{
        "recognized_from_project_history":bool(PRIOR_R2_TECHNICAL_ABORT_RECORDED),
        "r2_worker_sha256":PRIOR_R2_WORKER_SHA256,
        "reason":PRIOR_R2_ABORT_REASON,
        "usable_test_predictions":0,
        "test_derived_tuning":False,
        "physical_test_forward_upper_bound":PRIOR_R2_PHYSICAL_TEST_FORWARD_UPPER_BOUND,
        "mounted_prior_state":False,
    })
    print("[PRIOR LOCKED-TEST INPUT AUDIT] no mounted progress/shard state found; known R2 technical abort recorded in provenance.")

test_master=master[master.split.eq("test")].sort_values("official_row_index").reset_index(drop=True)
expected_test_idx=test_master.official_row_index.to_numpy(dtype=np.int64)
if len(expected_test_idx)!=932 or len(np.unique(expected_test_idx))!=932:
    raise RuntimeError("Locked TEST index identity drift before inference.")

test_gold_bin=np.asarray([
    [1 if str(r[t])=="Yes" else 0 for t in BINARY_TASKS]
    for r in test_master.to_dict("records")
],dtype=np.int64)
test_gold_lang=np.asarray([LANGUAGE_LABELS.index(str(x)) for x in test_master.language],dtype=np.int64)
test_gold_target=np.asarray([
    [1 if lab in set(json.loads(s)) else 0 for lab in TARGET_LABELS]
    for s in test_master.target_json
],dtype=np.int64)

def sigmoid_np(x):
    return 1.0/(1.0+np.exp(-np.clip(np.asarray(x,dtype=np.float64),-60,60)))

def softmax_np(x):
    x=np.asarray(x,dtype=np.float64)
    e=np.exp(x-x.max(axis=1,keepdims=True))
    return e/e.sum(axis=1,keepdims=True)

def binary_task_metrics(y,prob,thr):
    y=np.asarray(y,dtype=np.int64); prob=np.asarray(prob,dtype=float)
    pred=(prob>=float(thr)).astype(np.int64)
    p,r,f,s=precision_recall_fscore_support(y,pred,labels=[0,1],zero_division=0)
    return {
        "accuracy":float(accuracy_score(y,pred)),
        "macro_f1":float(f1_score(y,pred,average="macro",labels=[0,1],zero_division=0)),
        "threshold":float(thr),
        "classes":{
            "No":{"precision":float(p[0]),"recall":float(r[0]),"f1":float(f[0]),"support":int(s[0])},
            "Yes":{"precision":float(p[1]),"recall":float(r[1]),"f1":float(f[1]),"support":int(s[1])},
        },
        "confusion_matrix":confusion_matrix(y,pred,labels=[0,1]).astype(int).tolist(),
    },pred

def language_metrics(y,probs):
    y=np.asarray(y,dtype=np.int64); probs=np.asarray(probs,dtype=float)
    pred=probs.argmax(axis=1)
    p,r,f,s=precision_recall_fscore_support(y,pred,labels=[0,1,2],zero_division=0)
    return {
        "accuracy":float(accuracy_score(y,pred)),
        "macro_f1":float(f1_score(y,pred,average="macro",labels=[0,1,2],zero_division=0)),
        "classes":{
            LANGUAGE_LABELS[i]:{
                "precision":float(p[i]),"recall":float(r[i]),"f1":float(f[i]),"support":int(s[i])
            } for i in range(3)
        },
        "confusion_matrix":confusion_matrix(y,pred,labels=[0,1,2]).astype(int).tolist(),
    },pred

def target_metrics(y,probs,thresholds):
    y=np.asarray(y,dtype=np.int64); probs=np.asarray(probs,dtype=float)
    th=np.asarray([float(thresholds[x]) for x in TARGET_LABELS],dtype=float).reshape(1,-1)
    pred=(probs>=th).astype(np.int64)
    per={}
    f1s=[]; aps=[]
    for j,lab in enumerate(TARGET_LABELS):
        p,r,f,s=precision_recall_fscore_support(y[:,j],pred[:,j],labels=[1],zero_division=0)
        ap=float(average_precision_score(y[:,j],probs[:,j])) if len(np.unique(y[:,j]))>1 else float("nan")
        per[lab]={
            "precision":float(p[0]),"recall":float(r[0]),"f1":float(f[0]),
            "support":int(y[:,j].sum()),"ap":ap,"threshold":float(th[0,j]),
        }
        f1s.append(float(f[0])); aps.append(ap)
    none_gold=np.all(y==0,axis=1); none_pred=np.all(pred==0,axis=1)
    rare=[per["National"]["f1"],per["Religious"]["f1"]]
    return {
        "exact_set_accuracy":float(np.mean(np.all(y==pred,axis=1))),
        "macro_f1":float(np.mean(f1s)),
        "micro_f1":float(f1_score(y,pred,average="micro",zero_division=0)),
        "samples_f1":float(f1_score(y,pred,average="samples",zero_division=0)),
        "hamming_loss":float(hamming_loss(y,pred)),
        "map":float(np.nanmean(aps)),
        "rare_target_macro_f1":float(np.mean(rare)),
        "gold_none_support":int(none_gold.sum()),
        "predicted_none_count":int(none_pred.sum()),
        "none_exact_accuracy":float(np.mean(none_gold==none_pred)),
        "per_label":per,
    },pred

def core5_metrics(zb,zl,core_thresholds,mask=None):
    if mask is None: mask=np.ones(len(zb),dtype=bool)
    zb=np.asarray(zb)[mask]; zl=np.asarray(zl)[mask]
    yb=test_gold_bin[mask]; yl=test_gold_lang[mask]
    bp=sigmoid_np(zb); lp=softmax_np(zl)
    tasks={}; preds={}
    acc=[]; mf=[]
    for j,t in enumerate(BINARY_TASKS):
        m,pred=binary_task_metrics(yb[:,j],bp[:,j],core_thresholds[t])
        tasks[t]=m;preds[t]=pred;acc.append(m["accuracy"]);mf.append(m["macro_f1"])
    lm,lpred=language_metrics(yl,lp)
    tasks["language"]=lm;acc.append(lm["accuracy"]);mf.append(lm["macro_f1"])
    return {
        "accuracy":float(np.mean(acc)),
        "macro_f1":float(np.mean(mf)),
        "tasks":tasks,
    },bp,lp,preds,lpred

def load_test_shards(seed,feature_dir,bundle,parent):
    parts=[]; total_forwards=0
    for rank in [0,1]:
        npz=Path(feature_dir)/f"minicpm_facetgate_test_rank{rank}.npz"
        js=Path(feature_dir)/f"minicpm_facetgate_test_rank{rank}.json"
        if not npz.is_file() or not js.is_file():
            raise RuntimeError(f"Missing locked TEST shard seed={seed} rank={rank}")
        meta=load_json(js)
        if meta.get("schema")!="M3Offend_MiniCPM_V45_FacetGate_v1_0_1_LOCKED_TEST_GATED_FEATURE_SHARD_R3":
            raise RuntimeError(f"Wrong shard schema seed={seed} rank={rank}")
        if int(meta.get("seed",-1))!=seed or int(meta.get("rank",-1))!=rank or int(meta.get("world",-1))!=2:
            raise RuntimeError(f"Seed/rank/world drift seed={seed} rank={rank}")
        if meta.get("model_id")!=MODEL_ID or meta.get("model_revision")!=MODEL_REVISION:
            raise RuntimeError(f"Model identity drift seed={seed}")
        if meta.get("protocol_sha256")!=PROTOCOL_SHA256:
            raise RuntimeError(f"Protocol drift seed={seed}")
        if meta.get("parent_checkpoint_sha256")!=parent["sha256"]:
            raise RuntimeError(f"Gated parent SHA drift seed={seed}")
        if meta.get("shte_checkpoint_sha256")!=bundle["shte_checkpoint_sha256"]:
            raise RuntimeError(f"SHTE SHA drift seed={seed}")
        if int(meta.get("selected_internal_gated_step",-1))!=bundle["gated_step"]:
            raise RuntimeError(f"Gated step drift seed={seed}")
        if int(meta.get("selected_internal_shte_step",-1))!=bundle["shte_step"]:
            raise RuntimeError(f"SHTE step drift seed={seed}")
        if meta.get("test_identity_sha256")!=EXPECTED_TEST_IDENTITY_SHA256:
            raise RuntimeError(f"TEST identity drift seed={seed}")
        if not bool(meta.get("locked_test_authorized",False)) or not bool(meta.get("no_training",False)):
            raise RuntimeError(f"Locked TEST governance drift seed={seed}")
        if meta.get("feature_dtype")!="float32" or int(meta.get("representation_dim",-1))!=1024:
            raise RuntimeError(f"R3 float32 feature contract drift seed={seed} rank={rank}")
        if meta.get("specialist_execution")!="CPU_FLOAT32_AFTER_FEATURE_MERGE":
            raise RuntimeError(f"R3 specialist execution contract drift seed={seed} rank={rank}")
        if sha256_file(npz)!=meta.get("npz_sha256"):
            raise RuntimeError(f"Shard file SHA drift seed={seed} rank={rank}")
        total_forwards += int(meta.get("test_model_forwards",-1))
        with np.load(npz,allow_pickle=False) as z:
            parts.append({k:z[k] for k in z.files})

    keys=["official_row_index","image_id","visual","text","joint","z_binary","z_language","z_target_gated"]
    merged={k:np.concatenate([p[k] for p in parts],axis=0) for k in keys}
    order=np.argsort(merged["official_row_index"].astype(np.int64),kind="stable")
    merged={k:v[order] for k,v in merged.items()}

    if total_forwards!=932:
        raise RuntimeError(f"Seed {seed}: TEST forward count={total_forwards} != 932")
    if len(merged["official_row_index"])!=932 or len(np.unique(merged["official_row_index"]))!=932:
        raise RuntimeError(f"Seed {seed}: TEST row loss/duplication")
    if not np.array_equal(merged["official_row_index"].astype(np.int64),expected_test_idx):
        raise RuntimeError(f"Seed {seed}: ordered TEST identity mismatch")
    expected_names=test_master.image_id.astype(str).to_numpy()
    if not np.array_equal(merged["image_id"].astype(str),expected_names):
        raise RuntimeError(f"Seed {seed}: TEST image identity mismatch")
    return merged,total_forwards

def gold_language_strata(zb,zl,facet_probs,facet_thr):
    rows={}
    for lang in LANGUAGE_LABELS:
        li=LANGUAGE_LABELS.index(lang)
        mask=test_gold_lang==li
        c5,_,lp,_,lpred=core5_metrics(zb,zl,B_CURRENT_CORE_THRESHOLDS,mask)
        tm,_=target_metrics(test_gold_target[mask],facet_probs[mask],facet_thr)
        rows[lang]={
            "rows":int(mask.sum()),
            "core5_accuracy":c5["accuracy"],
            "core5_macro_f1":c5["macro_f1"],
            "language_accuracy":float(np.mean(lpred==test_gold_lang[mask])),
            "target_exact_set_accuracy":tm["exact_set_accuracy"],
            "target_macro_f1":tm["macro_f1"],
            "target_micro_f1":tm["micro_f1"],
            "target_samples_f1":tm["samples_f1"],
            "target_hamming_loss":tm["hamming_loss"],
            "target_map":tm["map"],
            "target_per_label":tm["per_label"],
            "binary_tasks":{t:c5["tasks"][t] for t in BINARY_TASKS},
        }
    return rows

PER_SEED_RESULTS=[]
FRESH_TEST_FORWARD_COUNT=0
VERIFIED_TEST_FORWARD_COUNT=0

for seed in PUBLICATION_SEEDS:
    print("\n"+"="*108)
    print(f"FINAL LOCKED TEST — MiniCPM-V-4.5 — FACETGATE v1.0.1 — SEED {seed}")
    print("="*108)

    bundle=BUNDLES[seed]
    parent=PARENTS[seed]
    score=bundle["scorecard"]
    thresholds=bundle["thresholds"]
    core_thr={k:float(v) for k,v in thresholds["core5"].items()}
    facet_thr={k:float(v) for k,v in thresholds["target"].items()}
    gated_target_thr={k:float(v) for k,v in thresholds["internal_gated_target_baseline"].items()}
    B_CURRENT_CORE_THRESHOLDS=core_thr  # consumed only by the reporting helper above

    seed_dir=WORK_ROOT/f"seed_{seed}"
    feature_dir=seed_dir/"locked_test_features"
    seed_dir.mkdir(parents=True,exist_ok=True)
    feature_dir.mkdir(parents=True,exist_ok=True)

    lock_payload={
        "architecture":"FacetGate",
        "protocol_sha256":PROTOCOL_SHA256,
        "test_identity_sha256":EXPECTED_TEST_IDENTITY_SHA256,
        "seed":seed,
        "model_id":MODEL_ID,
        "model_revision":MODEL_REVISION,
        "execution_mode":SCIENTIFIC_REGIME,
        "gated_step":bundle["gated_step"],
        "gated_checkpoint_sha256":parent["sha256"],
        "shte_step":bundle["shte_step"],
        "shte_checkpoint_sha256":bundle["shte_checkpoint_sha256"],
        "threshold_sha256":bundle["threshold_sha256"],
        "scorecard_sha256":bundle["scorecard_sha256"],
        "locked_test_worker_sha256":MINICPM_LOCKED_TEST_WORKER_SHA256,
    }
    lock_id=json_sha(lock_payload)
    marker=seed_dir/"LOCKED_TEST_ONCE_MARKER.json"
    metrics_path=seed_dir/"locked_test_metrics.json"
    ledger_path=seed_dir/"locked_test_prediction_ledger.parquet"

    reuse=False
    if marker.is_file():
        old=load_json(marker)
        if (
            old.get("lock_id")==lock_id and
            int(old.get("test_rows",-1))==932 and
            int(old.get("test_model_forwards",-1))==932 and
            metrics_path.is_file() and ledger_path.is_file() and
            sha256_file(metrics_path)==old.get("metrics_sha256") and
            sha256_file(ledger_path)==old.get("ledger_sha256")
        ):
            reuse=True
            print("[REUSE IDENTICAL COMPLETED LOCK] No repeated TEST forward",lock_id)

    if not reuse:
        # Fail closed on a partial TEST attempt. Never silently repeat TEST rows.
        existing=[p for p in feature_dir.iterdir()] if feature_dir.exists() else []
        if existing:
            raise RuntimeError(
                f"Seed {seed}: partial/mismatched locked-TEST files already exist without an identical verified completion marker. "
                "Automatic TEST rerun is forbidden. Audit the partial attempt before any further action."
            )

        cfg={
            "run_seed":seed,
            "model_id":MODEL_ID,
            "model_revision":MODEL_REVISION,
            "protocol_sha256":PROTOCOL_SHA256,
            "parent_gated_protocol_sha256":PARENT_GATED_PROTOCOL_SHA256,
            "execution_mode":SCIENTIFIC_REGIME,
            "parent_checkpoint":parent["path"],
            "parent_checkpoint_sha256":parent["sha256"],
            "parent_selected_step":parent["step"],
            "parent_thresholds":{},  # not consumed for TEST feature generation
            "shte_checkpoint":bundle["shte_checkpoint_path"],
            "shte_checkpoint_sha256":bundle["shte_checkpoint_sha256"],
            "shte_step":bundle["shte_step"],
            "prepared_manifest":str(PREPARED_MANIFEST),
            "feature_dir":str(feature_dir),
            "output_root":str(seed_dir/"worker_state"),
            "allow_locked_test":True,
            "expected_test_rows":932,
            "test_identity_sha256":EXPECTED_TEST_IDENTITY_SHA256,
            "task_instruction":TASK_INSTRUCTION,
            "binary_tasks":BINARY_TASKS,
            "language_labels":LANGUAGE_LABELS,
            "target_labels":TARGET_LABELS,
            "expected_hidden_size":EXPECTED_HIDDEN_SIZE,
            "expected_query_num":EXPECTED_QUERY_NUM,
            "max_ocr_tokens":MAX_OCR_TOKENS,
            "native_image_marker":NATIVE_IMAGE_MARKER,
            "native_chat_add_generation_prompt":NATIVE_CHAT_ADD_GENERATION_PROMPT,
            "native_chat_enable_thinking":NATIVE_CHAT_ENABLE_THINKING,
            "native_max_slice_nums":NATIVE_MAX_SLICE_NUMS,
            "native_max_input_length":NATIVE_MAX_INPUT_LENGTH,
            "bnb_4bit_skip_modules":BNB_4BIT_SKIP_MODULES,
            "resampler_batch_dim_status":RESAMPLER_BATCH_DIM_STATUS,
        }
        cfgp=seed_dir/"locked_test_worker_config.json"
        cfgp.write_text(json.dumps(cfg,indent=2),encoding="utf-8")

        env=os.environ.copy()
        env["FACETGATE_CONFIG"]=str(cfgp)
        env["TOKENIZERS_PARALLELISM"]="false"
        env["HF_HOME"]=str(HF_CACHE)
        cmd=[
            sys.executable,"-u","-m","torch.distributed.run",
            "--standalone","--nproc_per_node=2",str(WORKER_PATH)
        ]
        print("[LOCKED TEST WORKER]"," ".join(cmd),flush=True)
        t0=time.time()
        rc=subprocess.run(cmd,env=env,cwd=str(seed_dir)).returncode
        if rc!=0:
            state_files=sorted(str(p) for p in feature_dir.glob("*"))
            worker_logs=[]
            wlog=seed_dir/"worker_state"/"logs"
            if wlog.exists():
                worker_logs=sorted(str(p) for p in wlog.glob("*"))
            atomic_json(seed_dir/"LOCKED_TEST_FAILURE_DIAGNOSTIC.json",{
                "seed":seed,"returncode":rc,
                "feature_state_files":state_files,
                "worker_logs":worker_logs,
                "worker_sha256":MINICPM_LOCKED_TEST_WORKER_SHA256,
                "controller_revision":CONTROLLER_REVISION,
            })
            raise RuntimeError(
                f"Seed {seed} R3 locked-TEST feature worker failed rc={rc}. "
                f"Feature-state files={state_files}; worker logs={worker_logs}. "
                "The child traceback is printed inline above. "
                "Do NOT rerun automatically if any STARTED/PROGRESS marker exists."
            )
        print(f"[LOCKED TEST WORKER COMPLETE] seed={seed} seconds={time.time()-t0:.1f}",flush=True)

        M,seed_forwards=load_test_shards(seed,feature_dir,bundle,parent)
        FRESH_TEST_FORWARD_COUNT += seed_forwards
        VERIFIED_TEST_FORWARD_COUNT += seed_forwards

        # Exact successful-lock-test pattern used by InternVL3.5 / PaliGemma2 / Llama:
        # GPU worker emits Gated TEST representations/logits -> CPU float32 SHTE specialist.
        V=M["visual"].astype(np.float32)
        T=M["text"].astype(np.float32)
        J=M["joint"].astype(np.float32)
        ZB=M["z_binary"].astype(np.float32)
        ZL=M["z_language"].astype(np.float32)
        ZG=M["z_target_gated"].astype(np.float32)

        for name,arr,shape in [
            ("V",V,(932,1024)),("T",T,(932,1024)),("J",J,(932,1024)),
            ("ZB",ZB,(932,4)),("ZL",ZL,(932,3)),("ZG",ZG,(932,7)),
        ]:
            if tuple(arr.shape)!=shape or arr.dtype!=np.float32 or not np.isfinite(arr).all():
                raise RuntimeError(f"Seed {seed}: R3 merged float32 feature contract failed {name} shape={arr.shape} dtype={arr.dtype}")

        # Frozen SHTE is applied in CPU float32, matching MiniCPM development-cache semantics.
        st_shte=torch.load(bundle["shte_checkpoint_path"],map_location="cpu",weights_only=False)
        if not isinstance(st_shte,dict) or int(st_shte.get("step",-1))!=int(bundle["shte_step"]):
            raise RuntimeError(f"Seed {seed}: frozen SHTE checkpoint step/schema drift")
        if st_shte.get("protocol_sha256")!=PROTOCOL_SHA256:
            raise RuntimeError(f"Seed {seed}: frozen SHTE protocol SHA drift")
        if sha256_file(bundle["shte_checkpoint_path"])!=bundle["shte_checkpoint_sha256"]:
            raise RuntimeError(f"Seed {seed}: frozen SHTE checkpoint SHA drift")
        if not isinstance(st_shte.get("state_dict"),dict):
            raise RuntimeError(f"Seed {seed}: frozen SHTE checkpoint missing state_dict")

        specialist=FacetGateTarget().cpu().float().eval()
        specialist.load_state_dict(st_shte["state_dict"],strict=True)
        if any(p.dtype!=torch.float32 for p in specialist.parameters()):
            raise RuntimeError(f"Seed {seed}: specialist parameter dtype drift; expected float32")

        zstar_parts=[]; selector_parts=[]; view_parts=[]
        with torch.no_grad():
            for st in range(0,932,128):
                sl=slice(st,min(st+128,932))
                vv=torch.from_numpy(V[sl]).float()
                tt=torch.from_numpy(T[sl]).float()
                jj=torch.from_numpy(J[sl]).float()
                zz=torch.from_numpy(ZG[sl]).float()
                zS,aux=specialist(vv,tt,jj,zz)
                if zS.dtype!=torch.float32:
                    raise RuntimeError(f"Seed {seed}: SHTE output dtype drift {zS.dtype}")
                zstar_parts.append(zS.cpu().numpy().astype(np.float32))
                selector_parts.append(aux["selector"].cpu().numpy().astype(np.float32))
                view_parts.append(aux["view_weights"].cpu().numpy().astype(np.float32))
        ZS=np.concatenate(zstar_parts,axis=0)
        SEL=np.concatenate(selector_parts,axis=0)
        VW=np.concatenate(view_parts,axis=0)
        if ZS.shape!=(932,7) or SEL.shape!=(932,7) or VW.shape!=(932,4):
            raise RuntimeError(f"Seed {seed}: CPU specialist output shape drift ZS={ZS.shape} SEL={SEL.shape} VW={VW.shape}")
        del specialist,st_shte,zstar_parts,selector_parts,view_parts

        c5,bp,lp,bpred,lpred=core5_metrics(ZB,ZL,core_thr)
        pg=sigmoid_np(ZG)
        pf=sigmoid_np(ZS)
        gated_target,gated_target_pred=target_metrics(test_gold_target,pg,gated_target_thr)
        facet_target,facet_target_pred=target_metrics(test_gold_target,pf,facet_thr)

        full6_acc=float((5*c5["accuracy"]+facet_target["exact_set_accuracy"])/6.0)
        full6_mf1=float((5*c5["macro_f1"]+facet_target["macro_f1"])/6.0)

        language_strata=gold_language_strata(ZB,ZL,pf,facet_thr)

        selector_per_label={lab:float(SEL[:,j].mean()) for j,lab in enumerate(TARGET_LABELS)}
        selector_sd_per_label={lab:float(SEL[:,j].std()) for j,lab in enumerate(TARGET_LABELS)}
        view_mean=dict(zip(["visual","text","product","difference"],VW.mean(axis=0).tolist()))
        view_sd=dict(zip(["visual","text","product","difference"],VW.std(axis=0).tolist()))

        result={
            "schema":"M3Offend_MiniCPM_V45_FacetGate_v1_0_1_FINAL_LOCKED_TEST_SEED",
            "lock_id":lock_id,
            "architecture":"FacetGate",
            "internal_routes":{"core5":"Gated","target":"SHTE"},
            "seed":seed,
            "model_id":MODEL_ID,
            "model_revision":MODEL_REVISION,
            "protocol_version":"1.0.1",
            "protocol_sha256":PROTOCOL_SHA256,
            "test_rows":932,
            "test_identity_sha256":EXPECTED_TEST_IDENTITY_SHA256,
            "test_model_forwards":932,
            "frozen_development_binding":{
                "scorecard_sha256":bundle["scorecard_sha256"],
                "threshold_sha256":bundle["threshold_sha256"],
                "gated_step":bundle["gated_step"],
                "gated_checkpoint_sha256":parent["sha256"],
                "shte_step":bundle["shte_step"],
                "shte_checkpoint_sha256":bundle["shte_checkpoint_sha256"],
            },
            "facetgate":{
                "definition":"Internal Gated Core-5 route + internal SHTE Target specialist",
                "gated_core5_branch":{
                    "metrics":c5,
                    "selected_gated_step":bundle["gated_step"],
                    "checkpoint_sha256":parent["sha256"],
                },
                "shte_target_specialist":{
                    "metrics":facet_target,
                    "selected_step":bundle["shte_step"],
                    "checkpoint_sha256":bundle["shte_checkpoint_sha256"],
                    "selector_mean":float(SEL.mean()),
                    "selector_sd":float(SEL.std()),
                    "selector_per_label_mean":selector_per_label,
                    "selector_per_label_sd":selector_sd_per_label,
                    "view_weight_mean":view_mean,
                    "view_weight_sd":view_sd,
                },
                "full6_average_accuracy":full6_acc,
                "full6_macro_f1":full6_mf1,
            },
            "diagnostic_gated_target_baseline":{
                "metrics":gated_target,
                "target_macro_f1_delta_pp_to_facetgate":float(100*(facet_target["macro_f1"]-gated_target["macro_f1"])),
                "target_map_delta_to_facetgate":float(facet_target["map"]-gated_target["map"]),
            },
            "gold_language_stratified":language_strata,
            "no_test_training":True,
            "no_test_threshold_calibration":True,
            "no_test_checkpoint_selection":True,
            "no_test_result_based_rerun":True,
            "technical_retry_after_pre_metric_r2_abort":bool(PRIOR_R2_TECHNICAL_ABORT_RECORDED),
            "prior_r2_abort_reason":PRIOR_R2_ABORT_REASON,
            "prior_r2_usable_test_predictions":0,
            "specialist_execution":"CPU_FLOAT32_AFTER_GATED_FEATURE_MERGE",
        }

        # Full publication ledger.
        ledger=pd.DataFrame({
            "official_row_index":expected_test_idx,
            "image_id":test_master.image_id.astype(str).to_numpy(),
            "gold_language":[LANGUAGE_LABELS[i] for i in test_gold_lang],
            "seed":seed,
        })
        for j,t in enumerate(BINARY_TASKS):
            ledger[f"gold_{t}"]=test_gold_bin[:,j]
            ledger[f"facetgate_gated_prob_{t}"]=bp[:,j]
            ledger[f"facetgate_gated_pred_{t}"]=bpred[t]
            ledger[f"facetgate_gated_threshold_{t}"]=float(core_thr[t])
        ledger["facetgate_gated_pred_language"]=[LANGUAGE_LABELS[i] for i in lpred]
        for j,lab in enumerate(LANGUAGE_LABELS):
            ledger[f"facetgate_gated_prob_language_{lab}"]=lp[:,j]
        for j,lab in enumerate(TARGET_LABELS):
            ledger[f"gold_target_{lab}"]=test_gold_target[:,j]
            ledger[f"diagnostic_gated_target_prob_{lab}"]=pg[:,j]
            ledger[f"diagnostic_gated_target_pred_{lab}"]=gated_target_pred[:,j]
            ledger[f"diagnostic_gated_target_threshold_{lab}"]=float(gated_target_thr[lab])
            ledger[f"facetgate_target_prob_{lab}"]=pf[:,j]
            ledger[f"facetgate_target_pred_{lab}"]=facet_target_pred[:,j]
            ledger[f"facetgate_target_threshold_{lab}"]=float(facet_thr[lab])
            ledger[f"facetgate_selector_{lab}"]=SEL[:,j]
        for j,vn in enumerate(["visual","text","product","difference"]):
            ledger[f"facetgate_view_weight_{vn}"]=VW[:,j]
        ledger["facetgate_target_prediction_json"]=[
            json.dumps([lab for j,lab in enumerate(TARGET_LABELS) if row[j]],ensure_ascii=False)
            for row in facet_target_pred
        ]
        ledger["gold_target_json"]=[
            json.dumps([lab for j,lab in enumerate(TARGET_LABELS) if row[j]],ensure_ascii=False)
            for row in test_gold_target
        ]

        ledger.to_parquet(ledger_path,index=False)
        ledger.to_csv(seed_dir/"locked_test_prediction_ledger.csv",index=False)
        atomic_json(metrics_path,result)
        atomic_json(marker,{
            "schema":"M3Offend_MiniCPM_V45_FacetGate_TEST_ONCE_MARKER_v1_0_1",
            "lock_id":lock_id,
            "seed":seed,
            "test_rows":932,
            "test_identity_sha256":EXPECTED_TEST_IDENTITY_SHA256,
            "test_model_forwards":932,
            "ledger_sha256":sha256_file(ledger_path),
            "metrics_sha256":sha256_file(metrics_path),
            "protocol_sha256":PROTOCOL_SHA256,
            "no_test_result_based_rerun":True,
            "technical_retry_after_pre_metric_r2_abort":bool(PRIOR_R2_TECHNICAL_ABORT_RECORDED),
            "prior_r2_abort_reason":PRIOR_R2_ABORT_REASON,
            "prior_r2_usable_test_predictions":0,
            "specialist_execution":"CPU_FLOAT32_AFTER_GATED_FEATURE_MERGE",
        })
        del M,V,T,J,ZB,ZL,ZG,ZS,SEL,VW,bp,lp,pg,pf
        gc.collect()
        torch.cuda.empty_cache()
    else:
        VERIFIED_TEST_FORWARD_COUNT += 932

    result=load_json(metrics_path)
    if result.get("lock_id")!=lock_id or int(result.get("test_model_forwards",-1))!=932:
        raise RuntimeError(f"Seed {seed}: completed locked-TEST result identity drift.")
    PER_SEED_RESULTS.append(result)
    print("[FULL FACETGATE SEED COMPLETE]",{
        "seed":seed,
        "Core5_Acc":result["facetgate"]["gated_core5_branch"]["metrics"]["accuracy"],
        "Core5_MF1":result["facetgate"]["gated_core5_branch"]["metrics"]["macro_f1"],
        "Target_Exact":result["facetgate"]["shte_target_specialist"]["metrics"]["exact_set_accuracy"],
        "Target_MF1":result["facetgate"]["shte_target_specialist"]["metrics"]["macro_f1"],
        "Target_mAP":result["facetgate"]["shte_target_specialist"]["metrics"]["map"],
        "Full6_Acc":result["facetgate"]["full6_average_accuracy"],
        "Full6_MF1":result["facetgate"]["full6_macro_f1"],
        "TEST_rows":932,
    })

if VERIFIED_TEST_FORWARD_COUNT!=5*932:
    raise RuntimeError(f"Verified five-seed TEST coverage={VERIFIED_TEST_FORWARD_COUNT} != 4660")
print("[ALL FIVE MiniCPM-V-4.5 FACETGATE LOCKED TEST SEEDS COMPLETE]",{
    "verified_TEST_forwards":VERIFIED_TEST_FORWARD_COUNT,
    "fresh_TEST_forwards_this_execution":FRESH_TEST_FORWARD_COUNT,
    "no_TEST_derived_tuning":True,
    "prior_R2_technical_abort_recorded":bool(PRIOR_R2_TECHNICAL_ABORT_RECORDED),
    "official_R3_TEST_forwards":VERIFIED_TEST_FORWARD_COUNT,
})


In [ ]:

# CELL 8 — FIVE-SEED AGGREGATION + COMPLETE PAPER-READY OUTPUT PACKAGE
# CPU only. No additional TEST model forward.

import math, json, shutil, zipfile
import numpy as np, pandas as pd
from scipy.stats import t as student_t

SUMMARY_DIR=WORK_ROOT/"paper_ready"
SUMMARY_DIR.mkdir(parents=True,exist_ok=True)

def summarize(values):
    x=np.asarray(values,dtype=float)
    n=len(x); mean=float(np.nanmean(x))
    sd=float(np.nanstd(x,ddof=1)) if n>1 else 0.0
    valid=np.isfinite(x)
    nv=int(valid.sum())
    if nv>1:
        xv=x[valid]; crit=float(student_t.ppf(0.975,nv-1))
        sdev=float(xv.std(ddof=1)); half=crit*sdev/math.sqrt(nv)
        lo=float(xv.mean()-half); hi=float(xv.mean()+half)
    elif nv==1:
        lo=hi=float(x[valid][0])
    else:
        lo=hi=float("nan")
    return {
        "n":n,"valid_n":nv,"mean":mean,"sample_sd":sd,
        "ci95_low":lo,"ci95_high":hi,
        "min":float(np.nanmin(x)) if nv else float("nan"),
        "max":float(np.nanmax(x)) if nv else float("nan"),
    }

# ---------------- headline scalar table ----------------
scalar_rows=[]
gated_vs_rows=[]
binary_rows=[]
language_rows=[]
target_rows=[]
gold_lang_rows=[]
target_by_gold_lang_rows=[]
task_rows=[]
selector_rows=[]
view_rows=[]
confusions={}

for r in PER_SEED_RESULTS:
    seed=int(r["seed"])
    fg=r["facetgate"]
    c5=fg["gated_core5_branch"]["metrics"]
    tm=fg["shte_target_specialist"]["metrics"]
    gb=r["diagnostic_gated_target_baseline"]["metrics"]
    scalar_rows.append({
        "seed":seed,
        "core5_accuracy":c5["accuracy"],
        "core5_macro_f1":c5["macro_f1"],
        "gated_target_exact_set_accuracy":gb["exact_set_accuracy"],
        "gated_target_macro_f1":gb["macro_f1"],
        "gated_target_micro_f1":gb["micro_f1"],
        "gated_target_samples_f1":gb["samples_f1"],
        "gated_target_hamming_loss":gb["hamming_loss"],
        "gated_target_map":gb["map"],
        "facetgate_target_exact_set_accuracy":tm["exact_set_accuracy"],
        "facetgate_target_macro_f1":tm["macro_f1"],
        "facetgate_target_micro_f1":tm["micro_f1"],
        "facetgate_target_samples_f1":tm["samples_f1"],
        "facetgate_target_hamming_loss":tm["hamming_loss"],
        "facetgate_target_map":tm["map"],
        "facetgate_rare_target_macro_f1":tm["rare_target_macro_f1"],
        "target_macro_f1_delta_pp":r["diagnostic_gated_target_baseline"]["target_macro_f1_delta_pp_to_facetgate"],
        "target_map_delta":r["diagnostic_gated_target_baseline"]["target_map_delta_to_facetgate"],
        "full6_accuracy":fg["full6_average_accuracy"],
        "full6_macro_f1":fg["full6_macro_f1"],
        "selector_mean":fg["shte_target_specialist"]["selector_mean"],
        "selector_sd":fg["shte_target_specialist"]["selector_sd"],
    })
    gated_vs_rows.append({
        "seed":seed,
        "gated_target_macro_f1":gb["macro_f1"],
        "facetgate_target_macro_f1":tm["macro_f1"],
        "delta_macro_f1_pp":r["diagnostic_gated_target_baseline"]["target_macro_f1_delta_pp_to_facetgate"],
        "gated_target_map":gb["map"],
        "facetgate_target_map":tm["map"],
        "delta_map_pp":100.0*r["diagnostic_gated_target_baseline"]["target_map_delta_to_facetgate"],
        "gated_target_exact_set_accuracy":gb["exact_set_accuracy"],
        "facetgate_target_exact_set_accuracy":tm["exact_set_accuracy"],
    })

    # Task-level Core-5.
    for task,m in c5["tasks"].items():
        task_rows.append({
            "seed":seed,"task":task,"accuracy":m["accuracy"],"macro_f1":m["macro_f1"]
        })
    # Binary class-wise.
    for task in BINARY_TASKS:
        m=c5["tasks"][task]
        confusions.setdefault(str(seed),{})[task]=m["confusion_matrix"]
        for cls,cm in m["classes"].items():
            binary_rows.append({
                "seed":seed,"task":task,"class":cls,
                "precision":cm["precision"],"recall":cm["recall"],"f1":cm["f1"],"support":cm["support"],
                "accuracy":m["accuracy"],"task_macro_f1":m["macro_f1"],
            })
    # Language class-wise.
    lm=c5["tasks"]["language"]
    confusions.setdefault(str(seed),{})["language"]=lm["confusion_matrix"]
    for cls,cm in lm["classes"].items():
        language_rows.append({
            "seed":seed,"language_class":cls,
            "precision":cm["precision"],"recall":cm["recall"],"f1":cm["f1"],"support":cm["support"],
            "language_accuracy":lm["accuracy"],"language_macro_f1":lm["macro_f1"],
        })
    # Target label-wise, both baseline and FacetGate.
    for lab in TARGET_LABELS:
        gm=gb["per_label"][lab]; fm=tm["per_label"][lab]
        target_rows.append({
            "seed":seed,"target_label":lab,
            "support":fm["support"],
            "gated_precision":gm["precision"],"gated_recall":gm["recall"],"gated_f1":gm["f1"],"gated_ap":gm["ap"],
            "facetgate_precision":fm["precision"],"facetgate_recall":fm["recall"],"facetgate_f1":fm["f1"],"facetgate_ap":fm["ap"],
            "facetgate_threshold":fm["threshold"],"gated_threshold":gm["threshold"],
            "f1_delta_pp":100.0*(fm["f1"]-gm["f1"]),
            "ap_delta_pp":100.0*(fm["ap"]-gm["ap"]) if np.isfinite(fm["ap"]) and np.isfinite(gm["ap"]) else np.nan,
        })
    # Gold-language stratified outputs.
    for lang,gm in r["gold_language_stratified"].items():
        gold_lang_rows.append({
            "seed":seed,"gold_language":lang,"rows":gm["rows"],
            "core5_accuracy":gm["core5_accuracy"],"core5_macro_f1":gm["core5_macro_f1"],
            "language_accuracy":gm["language_accuracy"],
            "target_exact_set_accuracy":gm["target_exact_set_accuracy"],
            "target_macro_f1":gm["target_macro_f1"],
            "target_micro_f1":gm["target_micro_f1"],
            "target_samples_f1":gm["target_samples_f1"],
            "target_hamming_loss":gm["target_hamming_loss"],
            "target_map":gm["target_map"],
        })
        for lab in TARGET_LABELS:
            x=gm["target_per_label"][lab]
            target_by_gold_lang_rows.append({
                "seed":seed,"gold_language":lang,"target_label":lab,
                "precision":x["precision"],"recall":x["recall"],"f1":x["f1"],
                "support":x["support"],"ap":x["ap"],
            })
    # Selector / view statistics.
    sp=fg["shte_target_specialist"]
    for lab in TARGET_LABELS:
        selector_rows.append({
            "seed":seed,"target_label":lab,
            "selector_mean":sp["selector_per_label_mean"][lab],
            "selector_sd":sp["selector_per_label_sd"][lab],
        })
    for view,val in sp["view_weight_mean"].items():
        view_rows.append({
            "seed":seed,"view":view,
            "weight_mean":val,"weight_sd":sp["view_weight_sd"][view],
        })

scalar=pd.DataFrame(scalar_rows).sort_values("seed")
gated_vs=pd.DataFrame(gated_vs_rows).sort_values("seed")
binary_df=pd.DataFrame(binary_rows)
language_df=pd.DataFrame(language_rows)
target_df=pd.DataFrame(target_rows)
gold_lang_df=pd.DataFrame(gold_lang_rows)
target_gold_lang_df=pd.DataFrame(target_by_gold_lang_rows)
task_df=pd.DataFrame(task_rows)
selector_df=pd.DataFrame(selector_rows)
view_df=pd.DataFrame(view_rows)

def aggregate_group(df,group_cols,metric_cols):
    out=[]
    for key,g in df.groupby(group_cols,dropna=False,sort=True):
        if not isinstance(key,tuple): key=(key,)
        base={c:v for c,v in zip(group_cols,key)}
        for m in metric_cols:
            s=summarize(g[m].astype(float).to_numpy())
            for k,v in s.items():
                base[f"{m}_{k}"]=v
        out.append(base)
    return pd.DataFrame(out)

# Scalar five-seed mean ± SD + Student-t 95% CI.
scalar_summary=[]
for col in scalar.columns:
    if col=="seed": continue
    s=summarize(scalar[col].astype(float).to_numpy())
    scalar_summary.append({"metric":col,**s})
scalar_summary=pd.DataFrame(scalar_summary)

task_summary=aggregate_group(task_df,["task"],["accuracy","macro_f1"])
binary_summary=aggregate_group(binary_df,["task","class"],["precision","recall","f1","support","accuracy","task_macro_f1"])
language_summary=aggregate_group(language_df,["language_class"],["precision","recall","f1","support","language_accuracy","language_macro_f1"])
target_summary=aggregate_group(
    target_df,["target_label"],
    ["support","gated_precision","gated_recall","gated_f1","gated_ap",
     "facetgate_precision","facetgate_recall","facetgate_f1","facetgate_ap","f1_delta_pp","ap_delta_pp"]
)
gold_lang_summary=aggregate_group(
    gold_lang_df,["gold_language"],
    ["rows","core5_accuracy","core5_macro_f1","language_accuracy","target_exact_set_accuracy",
     "target_macro_f1","target_micro_f1","target_samples_f1","target_hamming_loss","target_map"]
)
target_gold_lang_summary=aggregate_group(
    target_gold_lang_df,["gold_language","target_label"],["precision","recall","f1","support","ap"]
)
gated_vs_summary=[]
for col in [c for c in gated_vs.columns if c!="seed"]:
    gated_vs_summary.append({"metric":col,**summarize(gated_vs[col].astype(float).to_numpy())})
gated_vs_summary=pd.DataFrame(gated_vs_summary)
selector_summary=aggregate_group(selector_df,["target_label"],["selector_mean","selector_sd"])
view_summary=aggregate_group(view_df,["view"],["weight_mean","weight_sd"])

# Publication-ready CSV package.
threshold_rows=[]
for seed in PUBLICATION_SEEDS:
    th=BUNDLES[seed]["thresholds"]
    row={"seed":seed,
         "selected_internal_gated_step":BUNDLES[seed]["gated_step"],
         "selected_internal_shte_step":BUNDLES[seed]["shte_step"],
         "validation_target_delta_pp":BUNDLES[seed]["scorecard"]["target_delta_pp"]}
    for k,v in th["core5"].items(): row[f"core5_threshold::{k}"]=float(v)
    for k,v in th["target"].items(): row[f"facetgate_target_threshold::{k}"]=float(v)
    for k,v in th["internal_gated_target_baseline"].items(): row[f"gated_target_threshold::{k}"]=float(v)
    threshold_rows.append(row)
threshold_df=pd.DataFrame(threshold_rows).sort_values("seed")

tables={
    "five_seed_per_seed_scalar.csv":scalar,
    "frozen_thresholds_and_selected_steps.csv":threshold_df,
    "five_seed_scalar_mean_sd_ci.csv":scalar_summary,
    "core5_task_per_seed.csv":task_df,
    "core5_task_mean_sd_ci.csv":task_summary,
    "binary_classwise_per_seed.csv":binary_df,
    "binary_classwise_mean_sd_ci.csv":binary_summary,
    "language_classwise_per_seed.csv":language_df,
    "language_classwise_mean_sd_ci.csv":language_summary,
    "target_label_per_seed.csv":target_df,
    "target_label_mean_sd_ci.csv":target_summary,
    "gold_language_stratified_per_seed.csv":gold_lang_df,
    "gold_language_stratified_mean_sd_ci.csv":gold_lang_summary,
    "target_by_gold_language_per_seed.csv":target_gold_lang_df,
    "target_by_gold_language_mean_sd_ci.csv":target_gold_lang_summary,
    "gated_vs_facetgate_per_seed.csv":gated_vs,
    "gated_vs_facetgate_mean_sd_ci.csv":gated_vs_summary,
    "selector_per_seed.csv":selector_df,
    "selector_mean_sd_ci.csv":selector_summary,
    "view_weights_per_seed.csv":view_df,
    "view_weights_mean_sd_ci.csv":view_summary,
}
for name,df in tables.items():
    df.to_csv(SUMMARY_DIR/name,index=False)

atomic_json(SUMMARY_DIR/"confusion_matrices_per_seed.json",confusions)

# Machine-readable complete five-seed result.
headline={
    "architecture":"FacetGate",
    "model_id":MODEL_ID,
    "model_revision":MODEL_REVISION,
    "protocol_version":"1.0.1",
    "controller_revision":CONTROLLER_REVISION,
    "protocol_sha256":PROTOCOL_SHA256,
    "publication_seeds":PUBLICATION_SEEDS,
    "test_rows_per_seed":932,
    "verified_total_TEST_forwards":4660,
    "fresh_TEST_forwards_this_execution":FRESH_TEST_FORWARD_COUNT,
    "test_identity_sha256":EXPECTED_TEST_IDENTITY_SHA256,
    "development_gate":DEV_GATE_REPORT,
    "scalar_summary":{r["metric"]:{k:v for k,v in r.items() if k!="metric"} for r in scalar_summary.to_dict("records")},
    "no_test_training":True,
    "no_test_threshold_calibration":True,
    "no_test_checkpoint_selection":True,
    "no_test_result_based_rerun":True,
    "prior_R2_technical_abort_recorded":bool(PRIOR_R2_TECHNICAL_ABORT_RECORDED),
    "prior_R2_usable_TEST_predictions":0,
    "official_R3_verified_TEST_forwards":4660,
}
atomic_json(SUMMARY_DIR/"FIVE_SEED_FINAL_LOCKED_TEST_SUMMARY.json",headline)

# Copy per-seed metrics and ledgers into one publication package while preserving originals.
for seed in PUBLICATION_SEEDS:
    src=WORK_ROOT/f"seed_{seed}"
    dst=SUMMARY_DIR/f"seed_{seed}"
    dst.mkdir(exist_ok=True)
    for name in [
        "locked_test_metrics.json","locked_test_prediction_ledger.csv",
        "locked_test_prediction_ledger.parquet","LOCKED_TEST_ONCE_MARKER.json"
    ]:
        p=src/name
        if p.is_file(): shutil.copy2(p,dst/name)

# Final immutable ZIP.
zip_path=WORK_ROOT/"M3Offend_9318_MiniCPM_V45_FacetGate_5Seed_FINAL_LOCKED_TEST_v1_0_1.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for p in SUMMARY_DIR.rglob("*"):
        if p.is_file(): z.write(p,p.relative_to(WORK_ROOT))
    for extra_name in ["FIVE_SEED_DEVELOPMENT_GATE.json","FROZEN_FIVE_SEED_BINDINGS.json"]:
        extra=WORK_ROOT/extra_name
        if extra.is_file(): z.write(extra,extra.relative_to(WORK_ROOT))

zip_sha=sha256_file(zip_path)
atomic_json(WORK_ROOT/"FINAL_LOCKED_TEST_COMPLETE.json",{
    "status":"ALL_FIVE_FINAL_LOCKED_TEST_COMPLETE",
    "architecture":"FacetGate",
    "model_id":MODEL_ID,
    "model_revision":MODEL_REVISION,
    "protocol_version":"1.0.1",
    "protocol_sha256":PROTOCOL_SHA256,
    "seeds":PUBLICATION_SEEDS,
    "test_rows_per_seed":932,
    "verified_total_TEST_forwards":4660,
    "test_identity_sha256":EXPECTED_TEST_IDENTITY_SHA256,
    "results_zip":str(zip_path),
    "results_zip_sha256":zip_sha,
    "no_test_training":True,
    "no_test_threshold_calibration":True,
    "no_test_checkpoint_selection":True,
    "no_test_result_based_rerun":True,
    "prior_R2_technical_abort_recorded":bool(PRIOR_R2_TECHNICAL_ABORT_RECORDED),
    "prior_R2_usable_TEST_predictions":0,
    "official_R3_verified_TEST_forwards":4660,
})

# Concise paper headline.
def sm(metric):
    row=scalar_summary[scalar_summary.metric.eq(metric)].iloc[0]
    return float(row["mean"]),float(row["sample_sd"])

c5a,c5a_sd=sm("core5_accuracy")
c5f,c5f_sd=sm("core5_macro_f1")
tea,tea_sd=sm("facetgate_target_exact_set_accuracy")
tef,tef_sd=sm("facetgate_target_macro_f1")
tem,tem_sd=sm("facetgate_target_map")
f6a,f6a_sd=sm("full6_accuracy")
f6f,f6f_sd=sm("full6_macro_f1")
d,d_sd=sm("target_macro_f1_delta_pp")

print("\n=== MiniCPM-V-4.5 FACETGATE v1.0.1 — FIVE-SEED FINAL LOCKED TEST ===")
print(f"Core-5 Accuracy     : {100*c5a:.2f} ± {100*c5a_sd:.2f}")
print(f"Core-5 Macro-F1     : {100*c5f:.2f} ± {100*c5f_sd:.2f}")
print(f"Target Exact-set Acc: {100*tea:.2f} ± {100*tea_sd:.2f}")
print(f"Target Macro-F1     : {100*tef:.2f} ± {100*tef_sd:.2f}")
print(f"Target mAP          : {100*tem:.2f} ± {100*tem_sd:.2f}")
print(f"Target MF1 Δ vs Gated: {d:+.2f} ± {d_sd:.2f} pp")
print(f"Full-6 Accuracy     : {100*f6a:.2f} ± {100*f6a_sd:.2f}")
print(f"Full-6 Macro-F1     : {100*f6f:.2f} ± {100*f6f_sd:.2f}")
print("Paper-ready language outputs:",SUMMARY_DIR/"language_classwise_mean_sd_ci.csv")
print("Gold-language stratified outputs:",SUMMARY_DIR/"gold_language_stratified_mean_sd_ci.csv")
print("Target-by-language outputs:",SUMMARY_DIR/"target_by_gold_language_mean_sd_ci.csv")
print("FINAL ZIP:",zip_path)
print("FINAL ZIP SHA256:",zip_sha)
